# Development Analysis and Configuration Selection

This notebook audits the completed development retrieval records, corrects the evidence-target representation where necessary, aggregates the 25 configurations and freezes one configuration per chunking strategy.

In [113]:
import pickle
from pathlib import Path
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from transformers import AutoTokenizer
import copy, gc, hashlib, os
import hashlib

import warnings
warnings.filterwarnings("ignore")  # suppress all warnings

In [117]:
PROJECT_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI/esg_rag_project")
RAG_AI_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI")
BACKUP_ROOT = RAG_AI_ROOT / "apollo_results_backup"
DEVELOPMENT_RESULTS_PATH = BACKUP_ROOT / "development_chunking_results_4cfb01ee22.jsonl"
DEVELOPMENT_CONFIG_PATH = BACKUP_ROOT / "development_chunking_configurations.csv"

assert DEVELOPMENT_RESULTS_PATH.exists(), f"Results not found: {DEVELOPMENT_RESULTS_PATH}"
assert DEVELOPMENT_CONFIG_PATH.exists(), f"Configurations not found: {DEVELOPMENT_CONFIG_PATH}"

In [23]:
def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

development_records = load_jsonl(DEVELOPMENT_RESULTS_PATH)
print("Results file:", DEVELOPMENT_RESULTS_PATH)
print("File size:", round(DEVELOPMENT_RESULTS_PATH.stat().st_size / 1024**2, 2), "MB")
print("All records:", len(development_records))

Results file: /Users/tanggiee/Desktop/RAG_AI/apollo_results_backup/development_chunking_results_4cfb01ee22.jsonl
File size: 19.17 MB
All records: 1425


In [24]:
# Completion markers identify configurations that finished successfully.
# Their metadata also prevents records from another corpus, model or experiment version
# from being mixed into the present analysis.
configuration_markers = [
    r for r in development_records
    if r.get("record_type") == "configuration_marker"
    and r.get("configuration_complete", False)
]

# A valid frozen experiment must use exactly one version, corpus and embedding model.
experiment_versions = {r.get("experiment_version") for r in configuration_markers}
corpus_hashes = {r.get("corpus_hash") for r in configuration_markers}
embedding_models = {r.get("embedding_model") for r in configuration_markers}

print("Completed markers:", len(configuration_markers))
print("Experiment versions:", experiment_versions)
print("Corpus hashes:", corpus_hashes)
print("Embedding models:", embedding_models)

assert len(configuration_markers) == 25
assert len(experiment_versions) == 1
assert len(corpus_hashes) == 1
assert len(embedding_models) == 1

# Store the verified identifiers for filtering all subsequent records.
EXPERIMENT_VERSION = next(iter(experiment_versions))
DEVELOPMENT_CORPUS_HASH = next(iter(corpus_hashes))
EMBEDDING_MODEL_NAME = next(iter(embedding_models))

print("Frozen experiment metadata verified.")

Completed markers: 25
Experiment versions: {'2.4.1'}
Corpus hashes: {'4cfb01ee22d2a6d039d64b1dc58be32815ce601aabbc01134cfcc130acaaeb59'}
Embedding models: {'BAAI/bge-m3'}
Frozen experiment metadata verified.


In [25]:
# Retain only completed query-level observations from the verified experiment.
query_records = [
    r for r in development_records
    if r.get("record_type") == "query_result"
    and r.get("experiment_version") == EXPERIMENT_VERSION
    and r.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH
    and r.get("embedding_model") == EMBEDDING_MODEL_NAME
]

development_results_df = pd.DataFrame(query_records)

# The design requires 25 configurations × 56 development questions.
# Every configuration-question pair must occur exactly once.
assert len(development_results_df) == 1400
assert development_results_df["configuration_id"].nunique() == 25
assert development_results_df.groupby("configuration_id")["qa_id"].nunique().eq(56).all()
assert not development_results_df.duplicated(["configuration_id", "qa_id"]).any()

print("Query results:", len(development_results_df))
print("Configurations:", development_results_df["configuration_id"].nunique())
print("Queries per configuration:", development_results_df.groupby("configuration_id")["qa_id"].nunique().unique())
print("Query-level development results verified.")

Query results: 1400
Configurations: 25
Queries per configuration: [56]
Query-level development results verified.


In [26]:
# Separate identifiers, numeric measurements and nested data before aggregation.
identifier_columns = [
    column for column in ["configuration_id", "method", "qa_id", "record_type",
                          "experiment_version", "corpus_hash", "embedding_model"]
    if column in development_results_df.columns
]
numeric_columns = development_results_df.select_dtypes(include=np.number).columns.tolist()
boolean_columns = development_results_df.select_dtypes(include="bool").columns.tolist()
other_columns = [
    column for column in development_results_df.columns
    if column not in identifier_columns + numeric_columns + boolean_columns
]

print("Identifier columns:")
print(identifier_columns)
print("\nNumeric metric columns:")
print(numeric_columns)
print("\nBoolean metric columns:")
print(boolean_columns)
print("\nOther stored columns:")
print(other_columns)

Identifier columns:
['configuration_id', 'method', 'qa_id', 'record_type', 'experiment_version', 'corpus_hash', 'embedding_model']

Numeric metric columns:
['coverage_at_1', 'unlocated_at_1', 'retrieval_seconds_at_1', 'coverage_at_3', 'unlocated_at_3', 'retrieval_seconds_at_3', 'coverage_at_5', 'unlocated_at_5', 'retrieval_seconds_at_5', 'coverage_at_10', 'unlocated_at_10', 'retrieval_seconds_at_10', 'reciprocal_rank_at_10', 'budget_retrieval_seconds', 'budget_coverage_ratio', 'budget_first_overlap_rank', 'budget_tokens_used', 'budget_results_used', 'budget_unlocated_results']

Boolean metric columns:
['configuration_complete', 'complete_recall_at_1', 'partial_recall_at_1', 'complete_recall_at_3', 'partial_recall_at_3', 'complete_recall_at_5', 'partial_recall_at_5', 'complete_recall_at_10', 'partial_recall_at_10', 'budget_complete_coverage', 'budget_partial_overlap']

Other stored columns:
['parameters', 'evidence_id', 'doc_id', 'retrieved_at_1', 'retrieved_at_3', 'retrieved_at_5', 're

In [27]:
# Show the structure of nested fields without filling the notebook with full text.
sample = query_records[0]
print("Sample configuration:", sample.get("configuration_id"))
print("Sample question:", sample.get("qa_id"))

for key, value in sample.items():
    if isinstance(value, list):
        description = f"list[{len(value)}]"
        if value and isinstance(value[0], dict): description += f", item keys={list(value[0])}"
    elif isinstance(value, dict):
        description = f"dict keys={list(value)}"
    else:
        description = f"{type(value).__name__}: {repr(value)[:120]}"
    print(f"{key}: {description}")

Sample configuration: sentence_3c2ed685
Sample question: 36_2016_TT-BTC_m_308617_article_0010_q01
record_type: str: 'query_result'
configuration_complete: bool: False
configuration_id: str: 'sentence_3c2ed685'
method: str: 'sentence'
parameters: dict keys=['chunk_size', 'chunk_overlap']
experiment_version: str: '2.4.1'
corpus_hash: str: '4cfb01ee22d2a6d039d64b1dc58be32815ce601aabbc01134cfcc130acaaeb59'
embedding_model: str: 'BAAI/bge-m3'
qa_id: str: '36_2016_TT-BTC_m_308617_article_0010_q01'
evidence_id: str: '36_2016_TT-BTC_m_308617_article_0010'
doc_id: str: '36_2016_TT-BTC_m_308617'
complete_recall_at_1: bool: False
partial_recall_at_1: bool: True
coverage_at_1: float: 0.08056159117499583
unlocated_at_1: int: 0
retrieval_seconds_at_1: float: 19.32420987499063
retrieved_at_1: list[1], item keys=['rank', 'node_id', 'doc_id', 'score', 'start_char', 'end_char', 'returned_tokens']
complete_recall_at_3: bool: False
partial_recall_at_3: bool: True
coverage_at_3: float: 0.218953702156109
un

## Validate retrieval metrics

The structural audit established that every configuration contains 56 unique query results. The following checks verify metric ranges, monotonic retrieval coverage, Boolean-label consistency and compliance with the fixed context-token budget.

In [28]:
# Coverage and reciprocal-rank values must remain within their mathematical ranges.
coverage_columns = [f"coverage_at_{k}" for k in [1, 3, 5, 10]]
recall_columns = [f"{label}_recall_at_{k}" for k in [1, 3, 5, 10] for label in ["complete", "partial"]]

assert development_results_df[coverage_columns].apply(lambda column: column.between(0, 1).all()).all()
assert development_results_df["reciprocal_rank_at_10"].between(0, 1).all()
assert development_results_df["budget_coverage_ratio"].between(0, 1).all()
assert development_results_df["budget_tokens_used"].between(0, 1000).all()
assert development_results_df[[f"retrieval_seconds_at_{k}" for k in [1, 3, 5, 10]] + ["budget_retrieval_seconds"]].ge(0).all().all()

# Adding more retrieved results cannot reduce cumulative gold-evidence coverage.
assert development_results_df["coverage_at_1"].le(development_results_df["coverage_at_3"] + 1e-12).all()
assert development_results_df["coverage_at_3"].le(development_results_df["coverage_at_5"] + 1e-12).all()
assert development_results_df["coverage_at_5"].le(development_results_df["coverage_at_10"] + 1e-12).all()

# Complete recall means full coverage; partial recall means that some gold evidence was recovered.
for k in [1, 3, 5, 10]:
    coverage = development_results_df[f"coverage_at_{k}"]
    assert development_results_df[f"complete_recall_at_{k}"].eq(coverage.ge(1 - 1e-12)).all()
    assert development_results_df[f"partial_recall_at_{k}"].eq(coverage.gt(0)).all()

assert development_results_df["budget_complete_coverage"].eq(development_results_df["budget_coverage_ratio"].ge(1 - 1e-12)).all()
assert development_results_df["budget_partial_overlap"].eq(development_results_df["budget_coverage_ratio"].gt(0)).all()

print("Metric-integrity checks passed.")
print("Unlocated fixed-k results:", int(development_results_df[[f"unlocated_at_{k}" for k in [1, 3, 5, 10]]].sum().sum()))
print("Unlocated budget results:", int(development_results_df["budget_unlocated_results"].sum()))

Metric-integrity checks passed.
Unlocated fixed-k results: 2846
Unlocated budget results: 702


In [29]:
# Inspect configuration-level records before deciding which efficiency fields to aggregate.
configuration_markers_df = pd.DataFrame(configuration_markers)

print("Completion-marker columns:")
print(configuration_markers_df.columns.tolist())

print("\nFirst completion marker:")
for key, value in configuration_markers[0].items():
    if isinstance(value, dict):
        description = f"dict keys={list(value)}"
    elif isinstance(value, list):
        description = f"list[{len(value)}]"
    else:
        description = f"{type(value).__name__}: {repr(value)[:150]}"
    print(f"{key}: {description}")

Completion-marker columns:
['record_type', 'configuration_id', 'configuration_complete', 'experiment_version', 'corpus_hash', 'embedding_model', 'method', 'parameters', 'indexed_nodes', 'total_nodes', 'chunking_seconds', 'indexing_seconds', 'node_checkpoint_path', 'embedding_checkpoint_path']

First completion marker:
record_type: str: 'configuration_marker'
configuration_id: str: 'sentence_3c2ed685'
configuration_complete: bool: True
experiment_version: str: '2.4.1'
corpus_hash: str: '4cfb01ee22d2a6d039d64b1dc58be32815ce601aabbc01134cfcc130acaaeb59'
embedding_model: str: 'BAAI/bge-m3'
method: str: 'sentence'
parameters: dict keys=['chunk_size', 'chunk_overlap']
indexed_nodes: int: 39306
total_nodes: int: 39306
chunking_seconds: float: 0.0
indexing_seconds: float: 29.230175042001065
node_checkpoint_path: str: '/Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/node_checkpoints/sentence_3c2ed685.pkl'
embedding_checkpoint_path: str: '/Users/tanggiee/Desktop/RAG_AI/esg_ra

In [30]:
# The manifest supplies the parameter values associated with each stable configuration ID.
development_configurations_df = pd.read_csv(DEVELOPMENT_CONFIG_PATH, encoding="utf-8-sig")

assert len(development_configurations_df) == 25
assert development_configurations_df["configuration_id"].is_unique
assert set(development_configurations_df["configuration_id"]) == set(development_results_df["configuration_id"])

print("Configurations:", len(development_configurations_df))
print("Configurations by method:")
print(development_configurations_df.groupby("method").size())
display(development_configurations_df)

Configurations: 25
Configurations by method:
method
hierarchical       5
semantic           5
sentence           5
sentence_window    5
token              5
dtype: int64


,configuration_id,method,parameters_json
0,sentence_3c2ed685,sentence,"{""chunk_overlap"": 20, ""chunk_size"": 128}"
1,sentence_4bb37dad,sentence,"{""chunk_overlap"": 30, ""chunk_size"": 256}"
2,sentence_b9110a00,sentence,"{""chunk_overlap"": 50, ""chunk_size"": 512}"
3,sentence_4abeca5b,sentence,"{""chunk_overlap"": 100, ""chunk_size"": 1024}"
4,sentence_2b17acfd,sentence,"{""chunk_overlap"": 200, ""chunk_size"": 2048}"
5,token_3c2ed685,token,"{""chunk_overlap"": 20, ""chunk_size"": 128}"
6,token_4bb37dad,token,"{""chunk_overlap"": 30, ""chunk_size"": 256}"
7,token_b9110a00,token,"{""chunk_overlap"": 50, ""chunk_size"": 512}"
8,token_4abeca5b,token,"{""chunk_overlap"": 100, ""chunk_size"": 1024}"
9,token_2b17acfd,token,"{""chunk_overlap"": 200, ""chunk_size"": 2048}"


#### Diagnose unlocated retrieval results

In [31]:
# Expand retrieved-node lists into a table so location metadata can be audited directly.
def build_node_audit(records, retrieved_field):
    rows = []
    for record in records:
        for node in record.get(retrieved_field, []):
            rows.append({
                "configuration_id": record["configuration_id"],
                "method": record["method"],
                "qa_id": record["qa_id"],
                "retrieval_field": retrieved_field,
                "rank": node.get("rank"),
                "node_id": node.get("node_id"),
                "doc_id": node.get("doc_id"),
                "start_char": node.get("start_char"),
                "end_char": node.get("end_char"),
                "returned_tokens": node.get("returned_tokens")
            })
    return pd.DataFrame(rows)

fixed_10_nodes_df = build_node_audit(query_records, "retrieved_at_10")
budget_nodes_df = build_node_audit(query_records, "budget_retrieved_nodes")

print("Fixed-k=10 nodes:", len(fixed_10_nodes_df))
print("Fixed-budget nodes:", len(budget_nodes_df))

Fixed-k=10 nodes: 13616
Fixed-budget nodes: 67147


In [32]:
# Classify the basic reason why a retrieved node cannot be mapped to source-text offsets.
def classify_location(row):
    if pd.isna(row["doc_id"]) or str(row["doc_id"]).strip() == "":
        return "missing_doc_id"
    if pd.isna(row["start_char"]) or pd.isna(row["end_char"]):
        return "missing_offsets"
    if not isinstance(row["start_char"], (int, float, np.integer, np.floating)):
        return "non_numeric_offsets"
    if not isinstance(row["end_char"], (int, float, np.integer, np.floating)):
        return "non_numeric_offsets"
    if row["start_char"] < 0 or row["end_char"] <= row["start_char"]:
        return "invalid_offset_range"
    return "valid_coordinates"

fixed_10_nodes_df["location_status"] = fixed_10_nodes_df.apply(classify_location, axis=1)
budget_nodes_df["location_status"] = budget_nodes_df.apply(classify_location, axis=1)

print("Fixed-k=10 location status:")
print(fixed_10_nodes_df["location_status"].value_counts())

print("\nFixed-budget location status:")
print(budget_nodes_df["location_status"].value_counts())

Fixed-k=10 location status:
location_status
valid_coordinates    10816
missing_offsets       2800
Name: count, dtype: int64

Fixed-budget location status:
location_status
valid_coordinates    53147
missing_offsets      14000
Name: count, dtype: int64


In [33]:
# Compare coordinate problems with the unlocated counts recorded during retrieval.
fixed_invalid_coordinates = fixed_10_nodes_df["location_status"].ne("valid_coordinates").sum()
budget_invalid_coordinates = budget_nodes_df["location_status"].ne("valid_coordinates").sum()
stored_unlocated_at_10 = development_results_df["unlocated_at_10"].sum()
stored_budget_unlocated = development_results_df["budget_unlocated_results"].sum()

print("Stored unlocated at k=10:", int(stored_unlocated_at_10))
print("Nodes with invalid coordinates at k=10:", int(fixed_invalid_coordinates))
print("Stored budget unlocated:", int(stored_budget_unlocated))
print("Budget nodes with invalid coordinates:", int(budget_invalid_coordinates))

Stored unlocated at k=10: 1335
Nodes with invalid coordinates at k=10: 2800
Stored budget unlocated: 702
Budget nodes with invalid coordinates: 14000


In [34]:
# Calculate the proportion of returned nodes recorded as unlocated for each configuration.
unlocated_rows = []

for configuration_id, group in development_results_df.groupby("configuration_id"):
    row = {
        "configuration_id": configuration_id,
        "method": group["method"].iloc[0],
        "parameters": group["parameters"].iloc[0]
    }

    for k in [1, 3, 5, 10]:
        returned = group[f"retrieved_at_{k}"].map(len).sum()
        row[f"unlocated_rate_at_{k}"] = group[f"unlocated_at_{k}"].sum() / returned if returned else 0

    budget_returned = group["budget_results_used"].sum()
    row["budget_unlocated_rate"] = group["budget_unlocated_results"].sum() / budget_returned if budget_returned else 0
    unlocated_rows.append(row)

unlocated_rates_df = pd.DataFrame(unlocated_rows).sort_values(
    ["budget_unlocated_rate", "unlocated_rate_at_10"],
    ascending=False
)

display(unlocated_rates_df)

,configuration_id,method,parameters,unlocated_rate_at_1,unlocated_rate_at_3,unlocated_rate_at_5,unlocated_rate_at_10,budget_unlocated_rate
19,sentence_window_ff7e26a2,sentence_window,{'window_size': 7},0.767857,0.607143,0.560714,0.478571,0.642857
17,sentence_window_ad4bb64a,sentence_window,{'window_size': 3},0.767857,0.601190,0.564286,0.476786,0.594714
18,sentence_window_d323da84,sentence_window,{'window_size': 5},0.785714,0.607143,0.567857,0.475000,0.592357
15,sentence_window_01a3c7e2,sentence_window,{'window_size': 2},0.767857,0.607143,0.564286,0.476786,0.531034
16,sentence_window_742e7319,sentence_window,{'window_size': 1},0.750000,0.601190,0.557143,0.476786,0.494824
0,hierarchical_24a6d460,hierarchical,"{'chunk_sizes': [2048, 512, 128]}",0.000000,0.000000,0.000000,0.000000,0.000000
1,hierarchical_521fae18,hierarchical,"{'chunk_sizes': [1024, 256, 64]}",0.000000,0.000000,0.000000,0.000000,0.000000
2,hierarchical_5b2e10d0,hierarchical,"{'chunk_sizes': [1024, 512, 256]}",0.000000,0.000000,0.000000,0.000000,0.000000
3,hierarchical_9c0a7874,hierarchical,"{'chunk_sizes': [2048, 1024, 512]}",0.000000,0.000000,0.000000,0.000000,0.000000
4,hierarchical_c2e0e660,hierarchical,"{'chunk_sizes': [4096, 1024, 256]}",0.000000,0.000000,0.000000,0.000000,0.000000


In [35]:
# Method-level means reveal whether the problem is concentrated in one chunking design.
method_unlocated_df = unlocated_rates_df.groupby("method").agg(
    configurations=("configuration_id", "count"),
    mean_unlocated_at_1=("unlocated_rate_at_1", "mean"),
    mean_unlocated_at_3=("unlocated_rate_at_3", "mean"),
    mean_unlocated_at_5=("unlocated_rate_at_5", "mean"),
    mean_unlocated_at_10=("unlocated_rate_at_10", "mean"),
    mean_budget_unlocated=("budget_unlocated_rate", "mean")
).reset_index()

display(method_unlocated_df)

,method,configurations,mean_unlocated_at_1,mean_unlocated_at_3,mean_unlocated_at_5,mean_unlocated_at_10,mean_budget_unlocated
0,hierarchical,5,0.000000,0.000000,0.000000,0.000000,0.000000
1,semantic,5,0.000000,0.000000,0.000000,0.000000,0.000000
2,sentence,5,0.000000,0.000000,0.000000,0.000000,0.000000
3,sentence_window,5,0.767857,0.604762,0.562857,0.476786,0.571157
4,token,5,0.000000,0.000000,0.000000,0.000000,0.000000


In [36]:
# Display representative problematic nodes without printing long retrieved text.
problem_nodes_df = pd.concat([
    fixed_10_nodes_df[fixed_10_nodes_df["location_status"] != "valid_coordinates"],
    budget_nodes_df[budget_nodes_df["location_status"] != "valid_coordinates"]
], ignore_index=True)

display(problem_nodes_df[
    ["configuration_id", "method", "qa_id", "retrieval_field", "rank",
     "node_id", "doc_id", "start_char", "end_char", "location_status"]
].head(30))

,configuration_id,method,qa_id,retrieval_field,rank,node_id,doc_id,start_char,end_char,location_status
0,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,1,53b3fb71-27f0-42c0-a2de-db5d3dc7cb15,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
1,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,2,4b864758-c844-4a26-b2b9-11e58ad29634,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
2,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,3,91ef57c9-a878-463b-a4be-4f1b87deaee4,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
3,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,4,a6512c4b-4c26-4905-ac76-2516a96bee93,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
4,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,5,b7d8e91a-d242-4803-b11e-485b3f02434b,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
5,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,6,d931fc8a-83db-4c3d-8667-9bca311060eb,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
6,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,7,fa3290ea-50cd-41b1-afaa-fa01d71b8d1b,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
7,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,8,24ac6241-d287-4203-81a8-c55d1996c757,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
8,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,9,7154979a-1c14-4adc-9e7c-e5d03c993ef5,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets
9,sentence_window_742e7319,sentence_window,36_2016_TT-BTC_m_308617_article_0010_q01,retrieved_at_10,10,c52d92bc-2465-45e8-8301-1302038749d1,36_2016_TT-BTC_m_308617,NaN,NaN,missing_offsets


In [42]:
# Search both the project folder and the Apollo backup for Sentence Window
# node checkpoints. No files are modified by this inspection.

BACKUP_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI/apollo_results_backup")

project_checkpoint_folder = (
    PROJECT_ROOT / "outputs" / "qa_benchmark" / "node_checkpoints"
)
backup_checkpoint_folder = BACKUP_ROOT / "node_checkpoints"

project_window_checkpoints = sorted(
    project_checkpoint_folder.glob("sentence_window_*.pkl")
)
backup_window_checkpoints = sorted(
    backup_checkpoint_folder.glob("sentence_window_*.pkl")
)

print("Project checkpoints:", len(project_window_checkpoints))
for path in project_window_checkpoints:
    print(" ", path.name, round(path.stat().st_size / 1024**2, 2), "MB")

print("\nBackup checkpoints:", len(backup_window_checkpoints))
for path in backup_window_checkpoints:
    print(" ", path.name, round(path.stat().st_size / 1024**2, 2), "MB")

Project checkpoints: 0

Backup checkpoints: 0


#### Fix Sentence_Window chunking method

In [45]:
# Inspect the smallest Sentence Window node checkpoint first.
# The checkpoint was created by our own experiment and is therefore trusted.
# Only one file is loaded to avoid unnecessary memory use.

BACKUP_CHECKPOINT_FOLDER = (Path("/Users/tanggiee/Desktop/RAG_AI") / "apollo_results_backup" / "node_checkpoints")

inspection_path = (BACKUP_CHECKPOINT_FOLDER / "sentence_window_742e7319.pkl")

assert inspection_path.exists(), f"Checkpoint not found: {inspection_path}"

with inspection_path.open("rb") as file:
    checkpoint_object = pickle.load(file)

print("Checkpoint:", inspection_path.name)
print("Top-level type:", type(checkpoint_object).__name__)

if isinstance(checkpoint_object, dict):
    print("Dictionary keys:", list(checkpoint_object))
elif isinstance(checkpoint_object, (list, tuple)):
    print("Top-level length:", len(checkpoint_object))
    if checkpoint_object:
        print("First-item type:", type(checkpoint_object[0]).__name__)

Checkpoint: sentence_window_742e7319.pkl
Top-level type: dict
Dictionary keys: ['metadata', 'indexed_nodes', 'all_nodes']


In [46]:
# Identify every list-like object that appears to contain LlamaIndex nodes.

node_collections = []

if isinstance(checkpoint_object, dict):
    for name, value in checkpoint_object.items():
        if isinstance(value, (list, tuple)) and value:
            first_item = value[0]
            if hasattr(first_item, "metadata") and hasattr(first_item, "node_id"):
                node_collections.append((str(name), value))

elif isinstance(checkpoint_object, (list, tuple)):
    # The checkpoint may directly contain nodes.
    if (
        checkpoint_object
        and hasattr(checkpoint_object[0], "metadata")
        and hasattr(checkpoint_object[0], "node_id")
    ):
        node_collections.append(("top_level", checkpoint_object))

    # Alternatively, it may contain separate indexed/all-node collections.
    else:
        for position, value in enumerate(checkpoint_object):
            if isinstance(value, (list, tuple)) and value:
                first_item = value[0]
                if hasattr(first_item, "metadata") and hasattr(first_item, "node_id"):
                    node_collections.append((f"item_{position}", value))

print("Node collections found:", len(node_collections))

for name, nodes in node_collections:
    print(
        name,
        "| nodes:", len(nodes),
        "| first type:", type(nodes[0]).__name__
    )

Node collections found: 1
indexed_nodes | nodes: 106509 | first type: TextNode


In [47]:
# Examine the provenance and window metadata preserved on one original node.

assert node_collections, "No node collection was found in the checkpoint."

collection_name, checkpoint_nodes = node_collections[0]
sample_node = checkpoint_nodes[0]

print("Collection:", collection_name)
print("Node ID:", sample_node.node_id)
print("Document ID:", sample_node.metadata.get("doc_id"))
print("Start offset:", sample_node.start_char_idx)
print("End offset:", sample_node.end_char_idx)
print("Metadata keys:", list(sample_node.metadata))

print("\nOriginal node text:")
print(repr(sample_node.get_content())[:1000])

print("\nOriginal-text metadata:")
print(repr(sample_node.metadata.get("original_text"))[:1000])

print("\nWindow metadata:")
print(repr(sample_node.metadata.get("window"))[:2000])

Collection: indexed_nodes
Node ID: 5afc3658-b05a-4912-856d-2179e0a54d29
Document ID: 38_2016_ND-CP_m_321908
Start offset: 0
End offset: 135
Metadata keys: ['doc_id', 'split', 'cleaned_filename', 'window', 'original_sentence']

Original node text:
'TABLE ROW | THE GOVERNMENT------- | THE SOCIALIST REPUBLIC OF VIETNAMIndependence - Freedom - Happiness---------------\nTABLE ROW | No. '

Original-text metadata:
None

Window metadata:
'TABLE ROW | THE GOVERNMENT------- | THE SOCIALIST REPUBLIC OF VIETNAMIndependence - Freedom - Happiness---------------\nTABLE ROW | No.  38/2016/ND-CP | Hanoi, May 15, 2016\nDECREE\nDETAILING A NUMBER OF ARTICLES OF THE LAW ON HYDRO-METEOROLOGY\nPursuant to the June 19, 2015 Law on Organization of the Government;\nPursuant to the November 23, 2015 Law on Hydro-meteorology;\nAt the proposal of the Minister of Natural Resources and Environment,\nThe Government promulgates the Decree detailing a number of articles of the Law on Hydro-meteorology.\n'


In [ ]:
# Validate the reconstruction strategy on the window_size=1 checkpoint.
# Each retrieved node ID should map back to an original anchor node with
# valid offsets. Its window should equal the neighbouring original sentences.
inspection_configuration_id = "sentence_window_742e7319"
inspection_window_size = 1
inspection_nodes = checkpoint_object["indexed_nodes"]

# Build direct node-ID lookup and document-level ordered sentence lists.
node_by_id = {node.node_id: node for node in inspection_nodes}
nodes_by_document = defaultdict(list)

for node in inspection_nodes:
    nodes_by_document[str(node.metadata["doc_id"])].append(node)

for doc_id in nodes_by_document:
    nodes_by_document[doc_id].sort(
        key=lambda node: (
            node.start_char_idx
            if node.start_char_idx is not None
            else float("inf")
        )
    )

print("Checkpoint nodes:", len(inspection_nodes))
print("Unique node IDs:", len(node_by_id))
print("Documents represented:", len(nodes_by_document))
print(
    "Nodes with valid anchor offsets:",
    sum(
        node.start_char_idx is not None
        and node.end_char_idx is not None
        for node in inspection_nodes
    )
)

Checkpoint nodes: 106509
Unique node IDs: 106509
Documents represented: 283
Nodes with valid anchor offsets: 106509


In [50]:
# Reconstruct each stored window from neighbouring original sentences.

window_reconstruction_rows = []

for doc_id, document_nodes in nodes_by_document.items():
    for position, anchor_node in enumerate(document_nodes):
        left = max(0, position - inspection_window_size)
        right = min(
            len(document_nodes),
            position + inspection_window_size + 1
        )

        neighbouring_nodes = document_nodes[left:right]

        reconstructed_window = " ".join(
            str(
                node.metadata.get(
                    "original_sentence",
                    node.get_content()
                )
            )
            for node in neighbouring_nodes
        )

        stored_window = str(anchor_node.metadata.get("window", ""))

        window_reconstruction_rows.append({
            "node_id": anchor_node.node_id,
            "doc_id": doc_id,
            "anchor_start": anchor_node.start_char_idx,
            "anchor_end": anchor_node.end_char_idx,
            "expanded_start": neighbouring_nodes[0].start_char_idx,
            "expanded_end": neighbouring_nodes[-1].end_char_idx,
            "window_exact_match": reconstructed_window == stored_window
        })

window_reconstruction_df = pd.DataFrame(window_reconstruction_rows)

print(
    "Exact reconstructed windows:",
    int(window_reconstruction_df["window_exact_match"].sum()),
    "of",
    len(window_reconstruction_df)
)
print(
    "Exact-match rate:",
    round(window_reconstruction_df["window_exact_match"].mean(), 6)
)
print(
    "Missing reconstructed offsets:",
    int(
        window_reconstruction_df[
            ["expanded_start", "expanded_end"]
        ].isna().any(axis=1).sum()
    )
)

Exact reconstructed windows: 106509 of 106509
Exact-match rate: 1.0
Missing reconstructed offsets: 0


In [51]:
# Collect every node ID saved for this configuration across fixed-k and
# fixed-budget retrieval. Repeated IDs are reduced to a unique set.

inspection_records = [
    record
    for record in query_records
    if record["configuration_id"] == inspection_configuration_id
]

saved_retrieved_node_ids = set()

for record in inspection_records:
    for k in [1, 3, 5, 10]:
        saved_retrieved_node_ids.update(
            node["node_id"]
            for node in record[f"retrieved_at_{k}"]
        )

    saved_retrieved_node_ids.update(
        node["node_id"]
        for node in record["budget_retrieved_nodes"]
    )

missing_checkpoint_node_ids = (
    saved_retrieved_node_ids - set(node_by_id)
)

print("Query records:", len(inspection_records))
print("Unique saved retrieved nodes:", len(saved_retrieved_node_ids))
print("Missing from checkpoint:", len(missing_checkpoint_node_ids))

Query records: 56
Unique saved retrieved nodes: 2517
Missing from checkpoint: 0


In [61]:
# Reconstruct the exact frozen 56-question development table used by
# notebook 05. This reads source data only and does not modify any file.

PARSED_FOLDER = PROJECT_ROOT / "data" / "parsed"
DEVELOPMENT_TEXT_FOLDER = PROJECT_ROOT / "data" / "splits" / "development"
BENCHMARK_FOLDER = PROJECT_ROOT / "outputs" / "qa_benchmark"

UNITS_PATH = PARSED_FOLDER / "development_units.jsonl"
PROVISIONS_PATH = PARSED_FOLDER / "development_provisions.jsonl"
DEVELOPMENT_QA_PATH = (
    BENCHMARK_FOLDER
    / "development_qa_generated_2.3.5-v3-direct-esg.jsonl"
)

for path in [UNITS_PATH, PROVISIONS_PATH, DEVELOPMENT_QA_PATH]:
    assert path.exists(), f"Required frozen input not found: {path}"

development_units_df = pd.DataFrame(load_jsonl(UNITS_PATH))
development_provisions_df = pd.DataFrame(load_jsonl(PROVISIONS_PATH))
all_development_qa_df = pd.DataFrame(load_jsonl(DEVELOPMENT_QA_PATH))

# Only the 56 manually approved records entered the retrieval experiment.
development_qa_df = all_development_qa_df[
    all_development_qa_df["usable"]
].copy()

print("Development units:", len(development_units_df))
print("Development provisions:", len(development_provisions_df))
print("Generated QA records:", len(all_development_qa_df))
print("Usable QA records:", len(development_qa_df))

Development units: 7847
Development provisions: 64576
Generated QA records: 60
Usable QA records: 56


In [62]:
# Standardise unit-level and provision-level passages into one gold lookup.
# Their original IDs become the evidence_id stored in the retrieval JSONL.

unit_gold_df = development_units_df.rename(columns={
    "unit_id": "evidence_id",
    "unit_text": "gold_evidence_text",
    "start_char": "gold_start_char",
    "end_char": "gold_end_char"
})

provision_gold_df = development_provisions_df.rename(columns={
    "provision_id": "evidence_id",
    "provision_text": "gold_evidence_text",
    "start_char": "gold_start_char",
    "end_char": "gold_end_char"
})

gold_columns = [
    "evidence_id",
    "doc_id",
    "cleaned_filename",
    "gold_evidence_text",
    "gold_start_char",
    "gold_end_char"
]

development_gold_df = pd.concat(
    [
        unit_gold_df[gold_columns],
        provision_gold_df[gold_columns]
    ],
    ignore_index=True
)

development_queries_df = development_qa_df.merge(
    development_gold_df,
    on=["evidence_id", "doc_id"],
    how="left",
    validate="one_to_one"
)

# Use the exact column names expected by the frozen retrieval scorer.
development_queries_df["original_evidence_text"] = (
    development_queries_df["gold_evidence_text"]
)
development_queries_df["evidence_start_char"] = pd.to_numeric(
    development_queries_df["gold_start_char"],
    errors="raise"
).astype(int)
development_queries_df["evidence_end_char"] = pd.to_numeric(
    development_queries_df["gold_end_char"],
    errors="raise"
).astype(int)

assert len(development_queries_df) == 56
assert development_queries_df["qa_id"].is_unique
assert development_queries_df[
    [
        "gold_evidence_text",
        "evidence_start_char",
        "evidence_end_char"
    ]
].notna().all().all()

print("Development queries reconstructed:", len(development_queries_df))

Development queries reconstructed: 56


In [63]:
# Each qa_id must use the same evidence_id and doc_id in all 25 configurations.

saved_query_identity_df = (
    development_results_df[
        ["qa_id", "evidence_id", "doc_id"]
    ]
    .astype(str)
    .drop_duplicates()
)

assert len(saved_query_identity_df) == 56, (
    "A qa_id has inconsistent evidence or document identifiers."
)

frozen_query_identity_df = development_queries_df[
    ["qa_id", "evidence_id", "doc_id"]
].astype(str)

identity_check_df = frozen_query_identity_df.merge(
    saved_query_identity_df,
    on=["qa_id", "evidence_id", "doc_id"],
    how="outer",
    indicator=True
)

print(identity_check_df["_merge"].value_counts())
assert identity_check_df["_merge"].eq("both").all()

print("The reconstructed gold table exactly matches the saved experiment.")

_merge
both          56
left_only      0
right_only     0
Name: count, dtype: int64
The reconstructed gold table exactly matches the saved experiment.


In [65]:
# Load the complete source texts required to validate the gold offsets.
# Plain text is sufficient; no embedding model or GPU is involved.

document_manifest_df = (
    development_units_df[
        ["doc_id", "cleaned_filename"]
    ]
    .drop_duplicates()
)

assert document_manifest_df["doc_id"].is_unique

document_text_by_id = {}

for row in document_manifest_df.itertuples(index=False):
    text_path = DEVELOPMENT_TEXT_FOLDER / row.cleaned_filename
    assert text_path.exists(), f"Source document not found: {text_path}"

    document_text_by_id[str(row.doc_id)] = text_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

offset_issues = []

for row in development_queries_df.itertuples(index=False):
    document_text = document_text_by_id[str(row.doc_id)]
    start = int(row.evidence_start_char)
    end = int(row.evidence_end_char)

    extracted_text = document_text[start:end].strip()
    expected_text = str(row.original_evidence_text).strip()

    if extracted_text != expected_text:
        offset_issues.append({
            "qa_id": row.qa_id,
            "evidence_id": row.evidence_id,
            "extracted_text": extracted_text,
            "expected_text": expected_text
        })

print("Corpus documents:", len(document_text_by_id))
print("Validated gold spans:", 56 - len(offset_issues))
print("Offset issues:", len(offset_issues))

assert not offset_issues, (
    "The reconstructed gold offsets do not match the source documents."
)

Corpus documents: 283
Validated gold spans: 56
Offset issues: 0


In [66]:
# Build corrected expanded-window provenance only for nodes that appeared
# in the saved retrieval results. This avoids retaining unnecessary mappings
# for all 106,509 indexed nodes.

def collect_saved_node_ids(records):
    """Collect unique node IDs from fixed-k and fixed-budget result lists."""
    node_ids = set()

    for record in records:
        for k in [1, 3, 5, 10]:
            node_ids.update(
                node["node_id"]
                for node in record[f"retrieved_at_{k}"]
            )

        node_ids.update(
            node["node_id"]
            for node in record["budget_retrieved_nodes"]
        )

    return node_ids


def build_window_provenance(indexed_nodes, requested_node_ids, window_size):
    """
    Reconstruct each requested node's returned Sentence Window and its
    corresponding source-document character range.
    """
    nodes_by_document = defaultdict(list)

    for node in indexed_nodes:
        doc_id = str(node.metadata["doc_id"])
        nodes_by_document[doc_id].append(node)

    # Sentence order within each document is defined by original offsets.
    for document_nodes in nodes_by_document.values():
        document_nodes.sort(key=lambda node: node.start_char_idx)

    provenance_by_node_id = {}
    source_mismatches = []

    for doc_id, document_nodes in nodes_by_document.items():
        document_text = document_text_by_id[doc_id]

        for position, anchor_node in enumerate(document_nodes):
            if anchor_node.node_id not in requested_node_ids:
                continue

            left = max(0, position - window_size)
            right = min(
                len(document_nodes),
                position + window_size + 1
            )

            neighbouring_nodes = document_nodes[left:right]
            sentence_texts = [
                str(node.metadata["original_sentence"])
                for node in neighbouring_nodes
            ]

            reconstructed_window = " ".join(sentence_texts)
            stored_window = str(anchor_node.metadata["window"])

            assert reconstructed_window == stored_window, (
                f"Window reconstruction failed: {anchor_node.node_id}"
            )

            # Confirm every constituent sentence still maps to its source span.
            for node, sentence_text in zip(
                neighbouring_nodes,
                sentence_texts
            ):
                source_text = document_text[
                    node.start_char_idx:node.end_char_idx
                ]

                if source_text != sentence_text:
                    source_mismatches.append({
                        "node_id": node.node_id,
                        "doc_id": doc_id,
                        "start_char": node.start_char_idx,
                        "end_char": node.end_char_idx
                    })

            provenance_by_node_id[anchor_node.node_id] = {
                "doc_id": doc_id,
                "expanded_start": neighbouring_nodes[0].start_char_idx,
                "expanded_end": neighbouring_nodes[-1].end_char_idx,
                "returned_text": reconstructed_window,
                "neighbouring_nodes": neighbouring_nodes
            }

    return provenance_by_node_id, source_mismatches

In [67]:
# Validate corrected provenance for the window_size=1 configuration.

inspection_records = [
    record
    for record in query_records
    if record["configuration_id"] == "sentence_window_742e7319"
]

requested_node_ids = collect_saved_node_ids(inspection_records)

window_provenance_by_node_id, source_mismatches = (
    build_window_provenance(
        checkpoint_object["indexed_nodes"],
        requested_node_ids,
        window_size=1
    )
)

missing_provenance_ids = (
    requested_node_ids
    - set(window_provenance_by_node_id)
)

print("Requested retrieved nodes:", len(requested_node_ids))
print("Reconstructed provenance:", len(window_provenance_by_node_id))
print("Missing provenance:", len(missing_provenance_ids))
print("Constituent source mismatches:", len(source_mismatches))

assert not missing_provenance_ids
assert not source_mismatches

print("Sentence Window provenance reconstruction passed.")

Requested retrieved nodes: 2517
Reconstructed provenance: 2517
Missing provenance: 0
Constituent source mismatches: 0
Sentence Window provenance reconstruction passed.


In [68]:
# Merge overlapping intervals so the same part of the gold evidence is not
# counted more than once when several retrieved windows overlap it.

def merge_intervals(intervals):
    """Combine overlapping character intervals."""
    if not intervals:
        return []

    sorted_intervals = sorted(intervals)
    merged = [list(sorted_intervals[0])]

    for start, end in sorted_intervals[1:]:
        previous_start, previous_end = merged[-1]

        if start <= previous_end:
            merged[-1][1] = max(previous_end, end)
        else:
            merged.append([start, end])

    return [tuple(interval) for interval in merged]


def trim_span_whitespace(document_text, start, end):
    """Remove leading and trailing whitespace from a gold character range."""
    while start < end and document_text[start].isspace():
        start += 1

    while end > start and document_text[end - 1].isspace():
        end -= 1

    return start, end

In [69]:
# Use qa_id to access the verified gold document and offsets during rescoring.

gold_query_by_qa_id = {
    str(row.qa_id): row
    for row in development_queries_df.itertuples(index=False)
}

assert len(gold_query_by_qa_id) == 56

In [70]:
# Score saved rankings using reconstructed expanded-window offsets.
# Retrieval is not repeated: node order and similarity scores remain frozen.

def score_saved_sentence_window(record, retrieved_nodes, provenance_lookup):
    """Calculate coverage from corrected Sentence Window source ranges."""
    gold_row = gold_query_by_qa_id[str(record["qa_id"])]
    gold_doc_id = str(gold_row.doc_id)
    document_text = document_text_by_id[gold_doc_id]

    gold_start, gold_end = trim_span_whitespace(
        document_text,
        int(gold_row.evidence_start_char),
        int(gold_row.evidence_end_char)
    )

    overlap_spans = []
    first_overlap_rank = None
    unlocated_results = 0

    for position, saved_node in enumerate(retrieved_nodes, start=1):
        # Wrong-document nodes cannot overlap the gold evidence, so their
        # character offsets are irrelevant to this question.
        if str(saved_node["doc_id"]) != gold_doc_id:
            continue

        provenance = provenance_lookup.get(saved_node["node_id"])

        if provenance is None:
            unlocated_results += 1
            continue

        assert provenance["doc_id"] == gold_doc_id

        returned_start = int(provenance["expanded_start"])
        returned_end = int(provenance["expanded_end"])

        if returned_start < gold_end and returned_end > gold_start:
            overlap_spans.append((
                max(returned_start, gold_start),
                min(returned_end, gold_end)
            ))

            if first_overlap_rank is None:
                first_overlap_rank = position

    merged_spans = merge_intervals(overlap_spans)
    covered_characters = sum(
        end - start
        for start, end in merged_spans
    )
    gold_length = gold_end - gold_start
    coverage_ratio = min(covered_characters / gold_length, 1.0)

    return {
        "complete_coverage": covered_characters >= gold_length,
        "partial_overlap": covered_characters > 0,
        "coverage_ratio": coverage_ratio,
        "reciprocal_rank": (
            1.0 / first_overlap_rank
            if first_overlap_rank is not None
            else 0.0
        ),
        "unlocated_results": unlocated_results
    }

In [71]:
# Rescore fixed k=1,3,5,10 for the first Sentence Window configuration.
# Keep old and corrected values side by side for transparent comparison.

fixed_k_comparison_rows = []

for record in inspection_records:
    comparison_row = {
        "configuration_id": record["configuration_id"],
        "qa_id": record["qa_id"]
    }

    for k in [1, 3, 5, 10]:
        corrected = score_saved_sentence_window(
            record,
            record[f"retrieved_at_{k}"],
            window_provenance_by_node_id
        )

        comparison_row[f"old_complete_at_{k}"] = bool(
            record[f"complete_recall_at_{k}"]
        )
        comparison_row[f"new_complete_at_{k}"] = corrected[
            "complete_coverage"
        ]

        comparison_row[f"old_partial_at_{k}"] = bool(
            record[f"partial_recall_at_{k}"]
        )
        comparison_row[f"new_partial_at_{k}"] = corrected[
            "partial_overlap"
        ]

        comparison_row[f"old_coverage_at_{k}"] = float(
            record[f"coverage_at_{k}"]
        )
        comparison_row[f"new_coverage_at_{k}"] = corrected[
            "coverage_ratio"
        ]

        comparison_row[f"old_unlocated_at_{k}"] = int(
            record[f"unlocated_at_{k}"]
        )
        comparison_row[f"new_unlocated_at_{k}"] = corrected[
            "unlocated_results"
        ]

        if k == 10:
            comparison_row["old_mrr_at_10"] = float(
                record["reciprocal_rank_at_10"]
            )
            comparison_row["new_mrr_at_10"] = corrected[
                "reciprocal_rank"
            ]

    fixed_k_comparison_rows.append(comparison_row)

fixed_k_comparison_df = pd.DataFrame(
    fixed_k_comparison_rows
)

assert len(fixed_k_comparison_df) == 56

In [72]:
# Summarise how corrected provenance changes retrieval quality.

fixed_k_summary_rows = []

for k in [1, 3, 5, 10]:
    fixed_k_summary_rows.append({
        "k": k,
        "old_complete_recall": fixed_k_comparison_df[
            f"old_complete_at_{k}"
        ].mean(),
        "corrected_complete_recall": fixed_k_comparison_df[
            f"new_complete_at_{k}"
        ].mean(),
        "old_partial_recall": fixed_k_comparison_df[
            f"old_partial_at_{k}"
        ].mean(),
        "corrected_partial_recall": fixed_k_comparison_df[
            f"new_partial_at_{k}"
        ].mean(),
        "old_mean_coverage": fixed_k_comparison_df[
            f"old_coverage_at_{k}"
        ].mean(),
        "corrected_mean_coverage": fixed_k_comparison_df[
            f"new_coverage_at_{k}"
        ].mean(),
        "old_unlocated": fixed_k_comparison_df[
            f"old_unlocated_at_{k}"
        ].sum(),
        "corrected_unlocated": fixed_k_comparison_df[
            f"new_unlocated_at_{k}"
        ].sum()
    })

fixed_k_summary_df = pd.DataFrame(fixed_k_summary_rows)

display(
    fixed_k_summary_df.style.format({
        "old_complete_recall": "{:.3f}",
        "corrected_complete_recall": "{:.3f}",
        "old_partial_recall": "{:.3f}",
        "corrected_partial_recall": "{:.3f}",
        "old_mean_coverage": "{:.3f}",
        "corrected_mean_coverage": "{:.3f}"
    })
)

print(
    "Old MRR@10:",
    round(fixed_k_comparison_df["old_mrr_at_10"].mean(), 3)
)
print(
    "Corrected MRR@10:",
    round(fixed_k_comparison_df["new_mrr_at_10"].mean(), 3)
)

assert fixed_k_summary_df["corrected_unlocated"].eq(0).all()

print("Corrected fixed-k scoring passed for window_size=1.")

,k,old_complete_recall,corrected_complete_recall,old_partial_recall,corrected_partial_recall,old_mean_coverage,corrected_mean_coverage,old_unlocated,corrected_unlocated
0,1,0.000,0.161,0.000,0.589,0.000,0.308,42,0
1,3,0.000,0.214,0.000,0.804,0.000,0.407,101,0
2,5,0.000,0.232,0.000,0.911,0.000,0.498,156,0
3,10,0.000,0.286,0.000,0.929,0.000,0.565,267,0


Old MRR@10: 0.0
Corrected MRR@10: 0.723
Corrected fixed-k scoring passed for window_size=1.


In [74]:
# Load only the tokenizer needed to reproduce the original 1,000-token
# truncation. No neural model weights or GPU are used.

from transformers import AutoTokenizer

bge_tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_MODEL_NAME,
    local_files_only=True
)

print("BGE tokenizer loaded; embedding model not loaded.")

BGE tokenizer loaded; embedding model not loaded.


In [75]:
# reproduce token truncation
def truncate_window_to_budget(text, token_budget):
    """Return the character endpoint allowed by the remaining token budget."""
    encoded = bge_tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True
    )

    offsets = encoded["offset_mapping"]

    if len(offsets) <= token_budget:
        return text, len(offsets), len(text)

    character_end = offsets[token_budget - 1][1]

    return text[:character_end], token_budget, character_end


def map_window_prefix_to_source_end(provenance, prefix_character_end):
    """
    Map the end of a truncated reconstructed window back to the corresponding
    character position in the original source document.
    """
    cursor = 0
    previous_source_end = provenance["expanded_start"]

    for position, node in enumerate(provenance["neighbouring_nodes"]):
        sentence_text = str(node.metadata["original_sentence"])
        sentence_start = cursor
        sentence_end = sentence_start + len(sentence_text)

        if prefix_character_end <= sentence_start:
            return previous_source_end

        if prefix_character_end < sentence_end:
            characters_inside_sentence = (
                prefix_character_end - sentence_start
            )
            return (
                node.start_char_idx
                + characters_inside_sentence
            )

        previous_source_end = node.end_char_idx
        cursor = sentence_end

        # Reconstructed windows use one inserted space between sentences.
        if position < len(provenance["neighbouring_nodes"]) - 1:
            cursor += 1

    return provenance["expanded_end"]

In [76]:
# correct the budget scorer
def score_saved_sentence_window_budget(
    record,
    provenance_lookup,
    token_budget=1000
):
    """Rescore a saved Sentence Window ranking under the fixed token budget."""
    gold_row = gold_query_by_qa_id[str(record["qa_id"])]
    gold_doc_id = str(gold_row.doc_id)
    document_text = document_text_by_id[gold_doc_id]

    gold_start, gold_end = trim_span_whitespace(
        document_text,
        int(gold_row.evidence_start_char),
        int(gold_row.evidence_end_char)
    )

    remaining_tokens = token_budget
    included_results = 0
    overlap_spans = []
    first_overlap_rank = None
    unlocated_results = 0

    for rank, saved_node in enumerate(
        record["budget_retrieved_nodes"],
        start=1
    ):
        if remaining_tokens == 0:
            break

        provenance = provenance_lookup.get(saved_node["node_id"])

        if provenance is None:
            # This should not occur because checkpoint coverage already passed.
            if str(saved_node["doc_id"]) == gold_doc_id:
                unlocated_results += 1
            continue

        returned_text = provenance["returned_text"]

        _, used_tokens, prefix_character_end = (
            truncate_window_to_budget(
                returned_text,
                remaining_tokens
            )
        )

        remaining_tokens -= used_tokens
        included_results += 1

        if str(saved_node["doc_id"]) != gold_doc_id:
            continue

        returned_start = provenance["expanded_start"]
        returned_end = map_window_prefix_to_source_end(
            provenance,
            prefix_character_end
        )

        if returned_start < gold_end and returned_end > gold_start:
            overlap_spans.append((
                max(returned_start, gold_start),
                min(returned_end, gold_end)
            ))

            if first_overlap_rank is None:
                first_overlap_rank = rank

    merged_spans = merge_intervals(overlap_spans)
    covered_characters = sum(
        end - start
        for start, end in merged_spans
    )
    gold_length = gold_end - gold_start
    coverage_ratio = min(covered_characters / gold_length, 1.0)

    return {
        "budget_complete_coverage": covered_characters >= gold_length,
        "budget_partial_overlap": covered_characters > 0,
        "budget_coverage_ratio": coverage_ratio,
        "budget_first_overlap_rank": first_overlap_rank,
        "budget_tokens_used": token_budget - remaining_tokens,
        "budget_results_used": included_results,
        "budget_unlocated_results": unlocated_results
    }

In [77]:
# Compare corrected budget metrics with the stored metrics while confirming
# that token use and the number of included results are reproduced exactly.

budget_comparison_rows = []

for record in inspection_records:
    corrected = score_saved_sentence_window_budget(
        record,
        window_provenance_by_node_id,
        token_budget=1000
    )

    budget_comparison_rows.append({
        "qa_id": record["qa_id"],
        "old_complete": record["budget_complete_coverage"],
        "new_complete": corrected["budget_complete_coverage"],
        "old_partial": record["budget_partial_overlap"],
        "new_partial": corrected["budget_partial_overlap"],
        "old_coverage": record["budget_coverage_ratio"],
        "new_coverage": corrected["budget_coverage_ratio"],
        "old_tokens": record["budget_tokens_used"],
        "new_tokens": corrected["budget_tokens_used"],
        "old_results": record["budget_results_used"],
        "new_results": corrected["budget_results_used"],
        "old_unlocated": record["budget_unlocated_results"],
        "new_unlocated": corrected["budget_unlocated_results"]
    })

budget_comparison_df = pd.DataFrame(budget_comparison_rows)

token_mismatches = budget_comparison_df[
    "old_tokens"
].ne(budget_comparison_df["new_tokens"]).sum()

result_count_mismatches = budget_comparison_df[
    "old_results"
].ne(budget_comparison_df["new_results"]).sum()

print("Token-use mismatches:", int(token_mismatches))
print("Result-count mismatches:", int(result_count_mismatches))
print(
    "Corrected unlocated results:",
    int(budget_comparison_df["new_unlocated"].sum())
)

assert token_mismatches == 0
assert result_count_mismatches == 0
assert budget_comparison_df["new_unlocated"].eq(0).all()

Token-use mismatches: 0
Result-count mismatches: 0
Corrected unlocated results: 0


In [78]:
budget_summary_df = pd.DataFrame([{
    "configuration_id": inspection_configuration_id,
    "old_complete_recall": budget_comparison_df["old_complete"].mean(),
    "corrected_complete_recall": budget_comparison_df["new_complete"].mean(),
    "old_partial_recall": budget_comparison_df["old_partial"].mean(),
    "corrected_partial_recall": budget_comparison_df["new_partial"].mean(),
    "old_mean_coverage": budget_comparison_df["old_coverage"].mean(),
    "corrected_mean_coverage": budget_comparison_df["new_coverage"].mean(),
    "old_unlocated": budget_comparison_df["old_unlocated"].sum(),
    "corrected_unlocated": budget_comparison_df["new_unlocated"].sum()
}])

display(
    budget_summary_df.style.format({
        "old_complete_recall": "{:.3f}",
        "corrected_complete_recall": "{:.3f}",
        "old_partial_recall": "{:.3f}",
        "corrected_partial_recall": "{:.3f}",
        "old_mean_coverage": "{:.3f}",
        "corrected_mean_coverage": "{:.3f}"
    })
)

print("Corrected fixed-budget scoring passed for window_size=1.")

,configuration_id,old_complete_recall,corrected_complete_recall,old_partial_recall,corrected_partial_recall,old_mean_coverage,corrected_mean_coverage,old_unlocated,corrected_unlocated
0,sentence_window_742e7319,0.000,0.250,0.000,0.929,0.000,0.527,239,0


Corrected fixed-budget scoring passed for window_size=1.


In [84]:
# Confirm that the correction covers exactly the five planned Sentence Window settings.
CORRECTION_VERSION = "sentence-window-provenance-v3"
sentence_window_markers = [m for m in configuration_markers if m.get("method") == "sentence_window"]
window_ids = [str(m["configuration_id"]) for m in sentence_window_markers]
window_sizes = [int(m["parameters"]["window_size"]) for m in sentence_window_markers]

assert len(sentence_window_markers) == 5
assert len(set(window_ids)) == 5
assert set(window_sizes) == {1, 2, 3, 5, 7}

for name in ["checkpoint_object", "inspection_nodes", "checkpoint_nodes", "node_by_id", "nodes_by_document", "window_provenance_by_node_id"]:
    globals().pop(name, None)
gc.collect()
print("Sentence Window configurations:", dict(sorted(zip(window_sizes, window_ids))))

Sentence Window configurations: {1: 'sentence_window_742e7319', 2: 'sentence_window_01a3c7e2', 3: 'sentence_window_ad4bb64a', 5: 'sentence_window_d323da84', 7: 'sentence_window_ff7e26a2'}


In [85]:
# Reconstruct and validate expanded-window provenance
def collect_saved_node_ids(records):
    """Collect unique node IDs used by fixed-k and fixed-budget retrieval."""
    node_ids = set()
    for record in records:
        for k in [1, 3, 5, 10]:
            node_ids.update(node["node_id"] for node in record[f"retrieved_at_{k}"])
        node_ids.update(node["node_id"] for node in record["budget_retrieved_nodes"])
    return node_ids


def build_window_provenance(indexed_nodes, requested_node_ids, window_size):
    """Reconstruct returned windows and their source-document character ranges."""
    nodes_by_document = defaultdict(list)
    for node in indexed_nodes:
        nodes_by_document[str(node.metadata["doc_id"])].append(node)
    for nodes in nodes_by_document.values():
        nodes.sort(key=lambda node: node.start_char_idx)

    provenance_by_node_id, source_issues, gap_diagnostics = {}, [], []
    for doc_id, document_nodes in nodes_by_document.items():
        document_text = document_text_by_id[doc_id]
        for position, anchor_node in enumerate(document_nodes):
            if anchor_node.node_id not in requested_node_ids:
                continue

            left = max(0, position - window_size)
            right = min(len(document_nodes), position + window_size + 1)
            neighbours = document_nodes[left:right]
            sentence_texts = [str(node.metadata["original_sentence"]) for node in neighbours]
            reconstructed_window = " ".join(sentence_texts)

            if reconstructed_window != str(anchor_node.metadata.get("window", "")):
                source_issues.append({"node_id": anchor_node.node_id, "issue": "window_mismatch"})

            for node, sentence_text in zip(neighbours, sentence_texts):
                source_text = document_text[node.start_char_idx:node.end_char_idx]
                if source_text != sentence_text:
                    source_issues.append({"node_id": anchor_node.node_id, "issue": "sentence_source_mismatch"})

            source_spans = [(int(node.start_char_idx), int(node.end_char_idx)) for node in neighbours]
            for left_node, right_node in zip(neighbours, neighbours[1:]):
                gap = document_text[left_node.end_char_idx:right_node.start_char_idx]
                if gap.strip():
                    gap_diagnostics.append({"node_id": anchor_node.node_id, "doc_id": doc_id, "gap_text": gap[:200]})

            provenance_by_node_id[anchor_node.node_id] = {
                "doc_id": doc_id,
                "expanded_start": int(neighbours[0].start_char_idx),
                "expanded_end": int(neighbours[-1].end_char_idx),
                "returned_text": reconstructed_window,
                "neighbouring_nodes": neighbours,
                "source_spans": source_spans,
                "has_non_whitespace_gap": any(document_text[a[1]:b[0]].strip() for a, b in zip(source_spans, source_spans[1:])),
            }
    return provenance_by_node_id, source_issues, gap_diagnostics


def correct_saved_node_spans(saved_nodes, provenance_lookup):
    """Copy audit nodes and replace only their Sentence Window source offsets."""
    corrected_nodes = []
    for saved_node in saved_nodes:
        corrected_node = copy.deepcopy(saved_node)
        provenance = provenance_lookup.get(saved_node["node_id"])
        if provenance is not None:
            corrected_node["start_char"] = provenance["expanded_start"]
            corrected_node["end_char"] = provenance["expanded_end"]
            corrected_node["source_spans"] = [list(span) for span in provenance["source_spans"]]
            corrected_node["span_representation"] = "ordered_source_spans"
        corrected_nodes.append(corrected_node)
    return corrected_nodes


def retrieval_signature(nodes):
    """Fields that must remain unchanged after provenance correction."""
    return [(n.get("rank"), n.get("node_id"), n.get("doc_id"), n.get("score"), n.get("returned_tokens")) for n in nodes]


def score_saved_sentence_window(record, retrieved_nodes, provenance_lookup):
    """Score fixed-k retrieval from the exact constituent source spans."""
    gold_row = gold_query_by_qa_id[str(record["qa_id"])]
    gold_doc_id = str(gold_row.doc_id)
    document_text = document_text_by_id[gold_doc_id]
    gold_start, gold_end = trim_span_whitespace(document_text, int(gold_row.evidence_start_char), int(gold_row.evidence_end_char))
    overlap_spans, first_overlap_rank, unlocated_results = [], None, 0

    for rank, saved_node in enumerate(retrieved_nodes, start=1):
        if str(saved_node["doc_id"]) != gold_doc_id:
            continue
        provenance = provenance_lookup.get(saved_node["node_id"])
        if provenance is None:
            unlocated_results += 1
            continue

        node_overlaps = []
        for start, end in provenance["source_spans"]:
            if start < gold_end and end > gold_start:
                node_overlaps.append((max(start, gold_start), min(end, gold_end)))
        overlap_spans.extend(node_overlaps)
        if node_overlaps and first_overlap_rank is None:
            first_overlap_rank = rank

    merged_spans = merge_intervals(overlap_spans)
    covered_characters = sum(end - start for start, end in merged_spans)
    gold_length = gold_end - gold_start
    return {
        "complete_coverage": covered_characters >= gold_length,
        "partial_overlap": covered_characters > 0,
        "coverage_ratio": min(covered_characters / gold_length, 1.0),
        "reciprocal_rank": 1.0 / first_overlap_rank if first_overlap_rank is not None else 0.0,
        "unlocated_results": unlocated_results,
    }


def map_window_prefix_to_source_spans(provenance, prefix_character_end):
    """Map a token-truncated window prefix to its exact constituent source spans."""
    mapped_spans, cursor = [], 0
    for position, node in enumerate(provenance["neighbouring_nodes"]):
        sentence_text = str(node.metadata["original_sentence"])
        sentence_start, sentence_end = cursor, cursor + len(sentence_text)
        included_characters = min(len(sentence_text), max(0, prefix_character_end - sentence_start))
        if included_characters > 0:
            mapped_spans.append((int(node.start_char_idx), int(node.start_char_idx + included_characters)))
        if prefix_character_end <= sentence_end:
            break
        cursor = sentence_end + (1 if position < len(provenance["neighbouring_nodes"]) - 1 else 0)
    return mapped_spans


def score_saved_sentence_window_budget(record, provenance_lookup, token_budget=1000):
    """Score the exact source spans returned under the frozen token budget."""
    gold_row = gold_query_by_qa_id[str(record["qa_id"])]
    gold_doc_id = str(gold_row.doc_id)
    document_text = document_text_by_id[gold_doc_id]
    gold_start, gold_end = trim_span_whitespace(document_text, int(gold_row.evidence_start_char), int(gold_row.evidence_end_char))
    remaining_tokens, included_results = token_budget, 0
    overlap_spans, first_overlap_rank, unlocated_results = [], None, 0

    for rank, saved_node in enumerate(record["budget_retrieved_nodes"], start=1):
        if remaining_tokens == 0:
            break
        provenance = provenance_lookup.get(saved_node["node_id"])
        if provenance is None:
            if str(saved_node["doc_id"]) == gold_doc_id:
                unlocated_results += 1
            continue

        _, used_tokens, prefix_end = truncate_window_to_budget(provenance["returned_text"], remaining_tokens)
        remaining_tokens -= used_tokens
        included_results += 1
        if str(saved_node["doc_id"]) != gold_doc_id:
            continue

        node_overlaps = []
        for start, end in map_window_prefix_to_source_spans(provenance, prefix_end):
            if start < gold_end and end > gold_start:
                node_overlaps.append((max(start, gold_start), min(end, gold_end)))
        overlap_spans.extend(node_overlaps)
        if node_overlaps and first_overlap_rank is None:
            first_overlap_rank = rank

    merged_spans = merge_intervals(overlap_spans)
    covered_characters = sum(end - start for start, end in merged_spans)
    gold_length = gold_end - gold_start
    coverage_ratio = min(covered_characters / gold_length, 1.0)
    return {
        "budget_complete_coverage": covered_characters >= gold_length,
        "budget_partial_overlap": covered_characters > 0,
        "budget_coverage_ratio": coverage_ratio,
        "budget_first_overlap_rank": first_overlap_rank,
        "budget_tokens_used": token_budget - remaining_tokens,
        "budget_results_used": included_results,
        "budget_unlocated_results": unlocated_results,
    }

In [86]:
# Correct each configuration with restart-safe per-configuration files
CORRECTION_CHECKPOINT_FOLDER = BACKUP_ROOT / "sentence_window_correction_checkpoints_v3"
CORRECTION_CHECKPOINT_FOLDER.mkdir(parents=True, exist_ok=True)


def write_json_atomic(path, payload):
    """Write complete JSON to a temporary file before atomically replacing the target."""
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False)
        file.flush()
        os.fsync(file.fileno())
    temporary_path.replace(path)


corrected_records_by_key, correction_summary_rows = {}, []

for marker in sorted(sentence_window_markers, key=lambda m: int(m["parameters"]["window_size"])):
    configuration_id = str(marker["configuration_id"])
    window_size = int(marker["parameters"]["window_size"])
    correction_path = CORRECTION_CHECKPOINT_FOLDER / f"{configuration_id}.json"
    original_records = [r for r in query_records if str(r["configuration_id"]) == configuration_id]
    assert len(original_records) == 56
    assert len({str(r["qa_id"]) for r in original_records}) == 56

    if correction_path.exists():
        payload = json.loads(correction_path.read_text(encoding="utf-8"))
        assert payload["correction_version"] == CORRECTION_VERSION
        assert payload["configuration_id"] == configuration_id
        assert payload["corpus_hash"] == DEVELOPMENT_CORPUS_HASH
        corrected_records = payload["records"]
        summary = payload["summary"]
        assert len(corrected_records) == 56
        print("Loaded correction checkpoint:", configuration_id)
    else:
        checkpoint_path = backup_checkpoint_folder / f"{configuration_id}.pkl"
        assert checkpoint_path.exists(), f"Missing node checkpoint: {checkpoint_path}"
        print("Processing:", configuration_id, "| window_size =", window_size)

        with checkpoint_path.open("rb") as file:
            checkpoint = pickle.load(file)

        requested_ids = collect_saved_node_ids(original_records)
        provenance_lookup, source_issues, gap_diagnostics = build_window_provenance(checkpoint["indexed_nodes"], requested_ids, window_size)
        assert requested_ids == set(provenance_lookup)
        assert not source_issues, source_issues[:5]
        print("Returned windows with non-whitespace source gaps:", len(gap_diagnostics))

        corrected_records = []
        for record in original_records:
            corrected_record = copy.deepcopy(record)

            for k in [1, 3, 5, 10]:
                scores = score_saved_sentence_window(record, record[f"retrieved_at_{k}"], provenance_lookup)
                corrected_record[f"complete_recall_at_{k}"] = scores["complete_coverage"]
                corrected_record[f"partial_recall_at_{k}"] = scores["partial_overlap"]
                corrected_record[f"coverage_at_{k}"] = scores["coverage_ratio"]
                corrected_record[f"unlocated_at_{k}"] = scores["unlocated_results"]
                corrected_record[f"retrieved_at_{k}"] = correct_saved_node_spans(record[f"retrieved_at_{k}"], provenance_lookup)
                if k == 10:
                    corrected_record["reciprocal_rank_at_10"] = scores["reciprocal_rank"]

                assert retrieval_signature(corrected_record[f"retrieved_at_{k}"]) == retrieval_signature(record[f"retrieved_at_{k}"])
                assert corrected_record[f"retrieval_seconds_at_{k}"] == record[f"retrieval_seconds_at_{k}"]

            budget_scores = score_saved_sentence_window_budget(record, provenance_lookup, token_budget=1000)
            assert budget_scores["budget_tokens_used"] == record["budget_tokens_used"]
            assert budget_scores["budget_results_used"] == record["budget_results_used"]

            corrected_record.update(budget_scores)
            corrected_record["budget_retrieved_nodes"] = correct_saved_node_spans(record["budget_retrieved_nodes"], provenance_lookup)
            corrected_record["sentence_window_provenance_corrected"] = True
            corrected_record["rescoring_version"] = CORRECTION_VERSION

            assert retrieval_signature(corrected_record["budget_retrieved_nodes"]) == retrieval_signature(record["budget_retrieved_nodes"])
            assert corrected_record["budget_retrieval_seconds"] == record["budget_retrieval_seconds"]
            corrected_records.append(corrected_record)

        corrected_df = pd.DataFrame(corrected_records)
        summary = {
            "configuration_id": configuration_id,
            "window_size": window_size,
            "retrieved_nodes_mapped": len(provenance_lookup),
            "windows_with_non_whitespace_gaps": len(gap_diagnostics),
            "complete_recall_at_10": float(corrected_df["complete_recall_at_10"].mean()),
            "partial_recall_at_10": float(corrected_df["partial_recall_at_10"].mean()),
            "mean_coverage_at_10": float(corrected_df["coverage_at_10"].mean()),
            "budget_complete_recall": float(corrected_df["budget_complete_coverage"].mean()),
            "budget_partial_recall": float(corrected_df["budget_partial_overlap"].mean()),
            "budget_mean_coverage": float(corrected_df["budget_coverage_ratio"].mean()),
        }
        payload = {"correction_version": CORRECTION_VERSION, "configuration_id": configuration_id, "corpus_hash": DEVELOPMENT_CORPUS_HASH, "records": corrected_records, "summary": summary, "gap_diagnostics": gap_diagnostics}
        write_json_atomic(correction_path, payload)
        print("Saved correction checkpoint:", correction_path.name)

        del checkpoint, provenance_lookup
        gc.collect()

    assert len({str(r["qa_id"]) for r in corrected_records}) == 56
    assert {str(r["qa_id"]) for r in corrected_records} == {str(r["qa_id"]) for r in original_records}
    for record in corrected_records:
        corrected_records_by_key[(configuration_id, str(record["qa_id"]))] = record
    correction_summary_rows.append(summary)

assert len(corrected_records_by_key) == 280
print("Corrected query records available:", len(corrected_records_by_key))

Processing: sentence_window_742e7319 | window_size = 1
Returned windows with non-whitespace source gaps: 0
Saved correction checkpoint: sentence_window_742e7319.json
Processing: sentence_window_01a3c7e2 | window_size = 2
Returned windows with non-whitespace source gaps: 0
Saved correction checkpoint: sentence_window_01a3c7e2.json
Processing: sentence_window_ad4bb64a | window_size = 3
Returned windows with non-whitespace source gaps: 2
Saved correction checkpoint: sentence_window_ad4bb64a.json
Processing: sentence_window_d323da84 | window_size = 5
Returned windows with non-whitespace source gaps: 7
Saved correction checkpoint: sentence_window_d323da84.json
Processing: sentence_window_ff7e26a2 | window_size = 7
Returned windows with non-whitespace source gaps: 9
Saved correction checkpoint: sentence_window_ff7e26a2.json
Corrected query records available: 280


In [87]:
# Display correction results
correction_summary_df = pd.DataFrame(correction_summary_rows).sort_values("window_size").reset_index(drop=True)
display(correction_summary_df.style.format({
    "complete_recall_at_10": "{:.3f}", "partial_recall_at_10": "{:.3f}", "mean_coverage_at_10": "{:.3f}",
    "budget_complete_recall": "{:.3f}", "budget_partial_recall": "{:.3f}", "budget_mean_coverage": "{:.3f}",
}))

,configuration_id,window_size,retrieved_nodes_mapped,windows_with_non_whitespace_gaps,complete_recall_at_10,partial_recall_at_10,mean_coverage_at_10,budget_complete_recall,budget_partial_recall,budget_mean_coverage
0,sentence_window_742e7319,1,2517,0,0.286,0.929,0.565,0.250,0.929,0.527
1,sentence_window_01a3c7e2,2,2517,0,0.393,0.929,0.672,0.304,0.911,0.573
2,sentence_window_ad4bb64a,3,2517,2,0.464,0.929,0.706,0.286,0.893,0.593
3,sentence_window_d323da84,5,2517,7,0.589,0.929,0.783,0.393,0.804,0.607
4,sentence_window_ff7e26a2,7,2517,9,0.661,0.929,0.811,0.411,0.696,0.570


In [88]:
# Assemble and atomically write a new corrected JSONL.
CORRECTED_RESULTS_PATH = BACKUP_ROOT / "development_chunking_results_4cfb01ee22_sentence_window_rescored_v3.jsonl"


def sha256_file(path):
    """Calculate a file hash to prove that the original JSONL remains unchanged."""
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


original_hash_before = sha256_file(DEVELOPMENT_RESULTS_PATH)
corrected_development_records, replaced_queries, updated_markers = [], 0, 0

for record in development_records:
    configuration_id = str(record.get("configuration_id"))
    qa_id = str(record.get("qa_id"))
    key = (configuration_id, qa_id)

    if record.get("record_type") == "query_result" and key in corrected_records_by_key:
        corrected_development_records.append(corrected_records_by_key[key])
        replaced_queries += 1
    elif record.get("record_type") == "configuration_marker" and configuration_id in set(window_ids):
        corrected_marker = copy.deepcopy(record)
        corrected_marker["sentence_window_provenance_corrected"] = True
        corrected_marker["rescoring_version"] = CORRECTION_VERSION
        corrected_development_records.append(corrected_marker)
        updated_markers += 1
    else:
        corrected_development_records.append(copy.deepcopy(record))

assert replaced_queries == 280
assert updated_markers == 5

temporary_path = CORRECTED_RESULTS_PATH.with_suffix(CORRECTED_RESULTS_PATH.suffix + ".tmp")
with temporary_path.open("w", encoding="utf-8") as file:
    for record in corrected_development_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
    file.flush()
    os.fsync(file.fileno())
temporary_path.replace(CORRECTED_RESULTS_PATH)

assert sha256_file(DEVELOPMENT_RESULTS_PATH) == original_hash_before
print("Original JSONL preserved:", DEVELOPMENT_RESULTS_PATH)
print("Corrected JSONL created:", CORRECTED_RESULTS_PATH)
print("Query records replaced:", replaced_queries)
print("Completion markers updated:", updated_markers)

Original JSONL preserved: /Users/tanggiee/Desktop/RAG_AI/apollo_results_backup/development_chunking_results_4cfb01ee22.jsonl
Corrected JSONL created: /Users/tanggiee/Desktop/RAG_AI/apollo_results_backup/development_chunking_results_4cfb01ee22_sentence_window_rescored_v3.jsonl
Query records replaced: 280
Completion markers updated: 5


In [89]:
# Final integrity audit
corrected_all_records = load_jsonl(CORRECTED_RESULTS_PATH)


def belongs_to_frozen_experiment(record):
    return (
        record.get("experiment_version") == EXPERIMENT_VERSION
        and record.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH
        and record.get("embedding_model") == EMBEDDING_MODEL_NAME
    )


corrected_queries = [r for r in corrected_all_records if r.get("record_type") == "query_result" and belongs_to_frozen_experiment(r)]
corrected_query_df = pd.DataFrame(corrected_queries)

assert len(corrected_query_df) == 1400
assert corrected_query_df["configuration_id"].nunique() == 25
assert corrected_query_df.groupby("configuration_id")["qa_id"].nunique().eq(56).all()
assert not corrected_query_df.duplicated(["configuration_id", "qa_id"]).any()

corrected_window_df = corrected_query_df[corrected_query_df["method"] == "sentence_window"]
other_corrected = {(str(r["configuration_id"]), str(r["qa_id"])): r for r in corrected_queries if r["method"] != "sentence_window"}
other_original = {(str(r["configuration_id"]), str(r["qa_id"])): r for r in query_records if r["method"] != "sentence_window"}

unlocated_columns = [f"unlocated_at_{k}" for k in [1, 3, 5, 10]] + ["budget_unlocated_results"]
remaining_unlocated = int(corrected_window_df[unlocated_columns].to_numpy().sum())
corrected_markers = [r for r in corrected_all_records if r.get("record_type") == "configuration_marker" and r.get("sentence_window_provenance_corrected") is True]

assert len(corrected_window_df) == 280
assert corrected_window_df["sentence_window_provenance_corrected"].eq(True).all()
assert remaining_unlocated == 0
assert other_corrected == other_original
assert len(other_corrected) == 1120
assert len(corrected_markers) == 5

print("Query records:", len(corrected_query_df))
print("Configurations:", corrected_query_df["configuration_id"].nunique())
print("Corrected Sentence Window records:", len(corrected_window_df))
print("Unchanged records from other methods:", len(other_corrected))
print("Remaining Sentence Window unlocated results:", remaining_unlocated)
print("Corrected completion markers:", len(corrected_markers))
print("Final corrected JSONL passed all integrity checks.")

Query records: 1400
Configurations: 25
Corrected Sentence Window records: 280
Unchanged records from other methods: 1120
Remaining Sentence Window unlocated results: 0
Corrected completion markers: 5
Final corrected JSONL passed all integrity checks.


In [94]:
# Load the corrected v3 file as the authoritative source for aggregation.
ANALYSIS_RESULTS_PATH = BACKUP_ROOT / "development_chunking_results_4cfb01ee22_sentence_window_rescored_v3.jsonl"
assert ANALYSIS_RESULTS_PATH.exists(), f"Corrected results not found: {ANALYSIS_RESULTS_PATH}"

analysis_records = load_jsonl(ANALYSIS_RESULTS_PATH)

def is_frozen_experiment(record):
    """Select only records from the verified development experiment."""
    return (
        record.get("experiment_version") == EXPERIMENT_VERSION
        and record.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH
        and record.get("embedding_model") == EMBEDDING_MODEL_NAME
    )

analysis_queries = [
    record for record in analysis_records
    if record.get("record_type") == "query_result"
    and is_frozen_experiment(record)
]

analysis_markers = [
    record for record in analysis_records
    if record.get("record_type") == "configuration_marker"
    and record.get("configuration_complete", False)
    and is_frozen_experiment(record)
]

analysis_query_df = pd.DataFrame(analysis_queries)
analysis_marker_df = pd.DataFrame(analysis_markers)

assert len(analysis_query_df) == 1400
assert len(analysis_marker_df) == 25
assert analysis_query_df["configuration_id"].nunique() == 25
assert analysis_query_df.groupby("configuration_id")["qa_id"].nunique().eq(56).all()
assert not analysis_query_df.duplicated(["configuration_id", "qa_id"]).any()

print("Analysis file:", ANALYSIS_RESULTS_PATH.name)
print("Query records:", len(analysis_query_df))
print("Completion markers:", len(analysis_marker_df))
print("Configurations:", analysis_query_df["configuration_id"].nunique())
print("Queries per configuration:", analysis_query_df.groupby("configuration_id")["qa_id"].nunique().unique())

Analysis file: development_chunking_results_4cfb01ee22_sentence_window_rescored_v3.jsonl
Query records: 1400
Completion markers: 25
Configurations: 25
Queries per configuration: [56]


## Aggregate and compare the 25 configurations

The corrected query-level records are aggregated into one row per configuration. Selection prioritises evidence coverage under the fixed 1,000-token budget; fixed-k quality and indexed-node workload provide supporting criteria.

In [97]:
# Calculate returned-token consumption at k=5 before aggregating.
analysis_query_df = analysis_query_df.copy()
analysis_query_df["returned_tokens_at_5"] = analysis_query_df["retrieved_at_5"].apply(
    lambda nodes: sum(int(node.get("returned_tokens") or 0) for node in nodes)
)

aggregation_spec = {
    "queries": ("qa_id", "nunique"),
    "partial_mrr_at_10": ("reciprocal_rank_at_10", "mean"),
    "mean_returned_tokens_at_5": ("returned_tokens_at_5", "mean"),
    "budget_complete_recall": ("budget_complete_coverage", "mean"),
    "budget_partial_recall": ("budget_partial_overlap", "mean"),
    "budget_mean_coverage": ("budget_coverage_ratio", "mean"),
    "budget_coverage_sd": ("budget_coverage_ratio", "std"),
    "median_budget_tokens": ("budget_tokens_used", "median"),
    "median_budget_results": ("budget_results_used", "median"),
    "median_budget_latency": ("budget_retrieval_seconds", "median"),
    "budget_unlocated_total": ("budget_unlocated_results", "sum"),
}

for k in [1, 3, 5, 10]:
    aggregation_spec.update({
        f"complete_recall_at_{k}": (f"complete_recall_at_{k}", "mean"),
        f"partial_recall_at_{k}": (f"partial_recall_at_{k}", "mean"),
        f"mean_coverage_at_{k}": (f"coverage_at_{k}", "mean"),
        f"coverage_sd_at_{k}": (f"coverage_at_{k}", "std"),
        f"median_latency_at_{k}": (f"retrieval_seconds_at_{k}", "median"),
        f"unlocated_total_at_{k}": (f"unlocated_at_{k}", "sum"),
    })

configuration_summary_df = (
    analysis_query_df.groupby(["configuration_id", "method"], as_index=False)
    .agg(**aggregation_spec)
)

assert len(configuration_summary_df) == 25
assert configuration_summary_df["queries"].eq(56).all()
print("Aggregated configurations:", len(configuration_summary_df))

Aggregated configurations: 25


In [98]:
# Attach configuration parameters and hardware-independent workload indicators.
marker_columns = [
    "configuration_id", "method", "parameters", "indexed_nodes",
    "total_nodes", "chunking_seconds", "indexing_seconds"
]
marker_summary_df = analysis_marker_df[marker_columns].copy()

assert len(marker_summary_df) == 25
assert marker_summary_df["configuration_id"].is_unique

marker_summary_df["parameters_json"] = marker_summary_df["parameters"].apply(
    lambda value: json.dumps(value, ensure_ascii=False, sort_keys=True)
)

configuration_summary_df = configuration_summary_df.merge(
    marker_summary_df,
    on=["configuration_id", "method"],
    how="left",
    validate="one_to_one"
)

required_metadata = ["parameters", "indexed_nodes", "total_nodes"]
assert configuration_summary_df[required_metadata].notna().all().all()

unlocated_columns = [
    "unlocated_total_at_1", "unlocated_total_at_3",
    "unlocated_total_at_5", "unlocated_total_at_10",
    "budget_unlocated_total"
]
configuration_summary_df["unlocated_total"] = configuration_summary_df[unlocated_columns].sum(axis=1)

assert configuration_summary_df["unlocated_total"].eq(0).all()
assert configuration_summary_df.groupby("method")["configuration_id"].count().eq(5).all()

print(configuration_summary_df["method"].value_counts().sort_index())
print("Remaining unlocated results:", int(configuration_summary_df["unlocated_total"].sum()))

method
hierarchical       5
semantic           5
sentence           5
sentence_window    5
token              5
Name: count, dtype: int64
Remaining unlocated results: 0


In [99]:
# Display configurations in the order used by the pre-specified hierarchy.
comparison_columns = [
    "method", "configuration_id", "parameters_json",
    "complete_recall_at_5", "partial_mrr_at_10", "complete_recall_at_10",
    "mean_returned_tokens_at_5", "median_latency_at_5", "indexed_nodes",
    "budget_complete_recall", "budget_partial_recall", "budget_mean_coverage"
]

configuration_comparison_df = configuration_summary_df[comparison_columns].sort_values(
    [
        "method", "complete_recall_at_5", "partial_mrr_at_10",
        "complete_recall_at_10", "mean_returned_tokens_at_5",
        "median_latency_at_5", "configuration_id"
    ],
    ascending=[True, False, False, False, True, True, True]
)

display(configuration_comparison_df.style.format({
    "complete_recall_at_5": "{:.3f}",
    "partial_mrr_at_10": "{:.3f}",
    "complete_recall_at_10": "{:.3f}",
    "mean_returned_tokens_at_5": "{:,.1f}",
    "median_latency_at_5": "{:.3f}",
    "indexed_nodes": "{:,.0f}",
    "budget_complete_recall": "{:.3f}",
    "budget_partial_recall": "{:.3f}",
    "budget_mean_coverage": "{:.3f}",
}))

,method,configuration_id,parameters_json,complete_recall_at_5,partial_mrr_at_10,complete_recall_at_10,mean_returned_tokens_at_5,median_latency_at_5,indexed_nodes,budget_complete_recall,budget_partial_recall,budget_mean_coverage
3,hierarchical,hierarchical_9c0a7874,"{""chunk_sizes"": [2048, 1024, 512]}",0.500,0.594,0.571,"2,426.4",0.392,"11,118",0.179,0.321,0.255
4,hierarchical,hierarchical_c2e0e660,"{""chunk_sizes"": [4096, 1024, 256]}",0.304,0.679,0.464,"1,262.1",0.716,"20,195",0.179,0.500,0.351
2,hierarchical,hierarchical_5b2e10d0,"{""chunk_sizes"": [1024, 512, 256]}",0.268,0.642,0.321,"1,215.6",0.802,"22,568",0.232,0.696,0.476
0,hierarchical,hierarchical_24a6d460,"{""chunk_sizes"": [2048, 512, 128]}",0.232,0.686,0.286,609.4,1.503,"42,147",0.232,0.750,0.474
1,hierarchical,hierarchical_521fae18,"{""chunk_sizes"": [1024, 256, 64]}",0.089,0.772,0.107,304.6,3.319,"92,722",0.125,0.893,0.422
8,semantic,semantic_b61156c9,"{""buffer_size"": 1, ""threshold"": 98}",0.643,0.572,0.714,"16,765.4",0.107,"2,557",0.143,0.214,0.181
6,semantic,semantic_4a84176f,"{""buffer_size"": 3, ""threshold"": 95}",0.571,0.565,0.661,"7,571.7",0.211,"5,713",0.179,0.375,0.315
9,semantic,semantic_b70aa92a,"{""buffer_size"": 1, ""threshold"": 95}",0.554,0.554,0.696,"7,388.3",0.207,"5,713",0.214,0.446,0.335
5,semantic,semantic_49151447,"{""buffer_size"": 2, ""threshold"": 95}",0.482,0.523,0.607,"7,157.9",0.209,"5,712",0.196,0.375,0.295
7,semantic,semantic_512278be,"{""buffer_size"": 1, ""threshold"": 90}",0.429,0.553,0.518,"3,737.3",0.389,"11,006",0.179,0.625,0.418


## Paired bootstrap selection

Within each method, the configuration with the highest observed complete Recall@5 was used as the reference. Paired bootstrap resampling preserved question-level pairing. Configurations whose 95% confidence interval for the reference-minus-candidate difference included zero were retained as statistically competitive. The pre-specified secondary criteria were then applied within that candidate set.

In [104]:
# Compare configurations on the same 56 questions using paired resampling.
BOOTSTRAP_REPLICATES = 10_000
BOOTSTRAP_SEED = 20260908
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []

for method, method_summary in configuration_summary_df.groupby("method"):
    best_row = method_summary.sort_values(
        ["complete_recall_at_5", "configuration_id"],
        ascending=[False, True]
    ).iloc[0]
    reference_id = best_row["configuration_id"]

    method_queries = analysis_query_df[analysis_query_df["method"] == method]
    pivot = method_queries.pivot(
        index="qa_id",
        columns="configuration_id",
        values="complete_recall_at_5"
    ).astype(float)

    assert pivot.shape == (56, 5)
    assert not pivot.isna().any().any()

    reference = pivot[reference_id].to_numpy()

    for candidate_id in pivot.columns:
        candidate = pivot[candidate_id].to_numpy()
        paired_difference = reference - candidate
        sample_indices = rng.integers(0, len(paired_difference), size=(BOOTSTRAP_REPLICATES, len(paired_difference)))
        bootstrap_differences = paired_difference[sample_indices].mean(axis=1)
        ci_low, ci_high = np.quantile(bootstrap_differences, [0.025, 0.975])

        bootstrap_rows.append({
            "method": method,
            "reference_id": reference_id,
            "candidate_id": candidate_id,
            "reference_recall_at_5": reference.mean(),
            "candidate_recall_at_5": candidate.mean(),
            "observed_difference": paired_difference.mean(),
            "ci_low": ci_low,
            "ci_high": ci_high,
            "statistically_competitive": ci_low <= 0 <= ci_high,
        })

bootstrap_comparison_df = pd.DataFrame(bootstrap_rows)
assert len(bootstrap_comparison_df) == 25
display(bootstrap_comparison_df.style.format({
    "reference_recall_at_5": "{:.3f}",
    "candidate_recall_at_5": "{:.3f}",
    "observed_difference": "{:.3f}",
    "ci_low": "{:.3f}",
    "ci_high": "{:.3f}",
}))

,method,reference_id,candidate_id,reference_recall_at_5,candidate_recall_at_5,observed_difference,ci_low,ci_high,statistically_competitive
0,hierarchical,hierarchical_9c0a7874,hierarchical_24a6d460,0.500,0.232,0.268,0.125,0.411,False
1,hierarchical,hierarchical_9c0a7874,hierarchical_521fae18,0.500,0.089,0.411,0.268,0.554,False
2,hierarchical,hierarchical_9c0a7874,hierarchical_5b2e10d0,0.500,0.268,0.232,0.125,0.339,False
3,hierarchical,hierarchical_9c0a7874,hierarchical_9c0a7874,0.500,0.500,0.000,0.000,0.000,True
4,hierarchical,hierarchical_9c0a7874,hierarchical_c2e0e660,0.500,0.304,0.196,0.071,0.321,False
5,semantic,semantic_b61156c9,semantic_49151447,0.643,0.482,0.161,0.054,0.268,False
6,semantic,semantic_b61156c9,semantic_4a84176f,0.643,0.571,0.071,-0.036,0.179,True
7,semantic,semantic_b61156c9,semantic_512278be,0.643,0.429,0.214,0.071,0.357,False
8,semantic,semantic_b61156c9,semantic_b61156c9,0.643,0.643,0.000,0.000,0.000,True
9,semantic,semantic_b61156c9,semantic_b70aa92a,0.643,0.554,0.089,0.000,0.179,True


In [102]:
# Retain bootstrap-competitive configurations, then apply the frozen tie-breakers.
competitive_ids = set(
    bootstrap_comparison_df.loc[
        bootstrap_comparison_df["statistically_competitive"],
        "candidate_id"
    ]
)

competitive_configurations_df = configuration_summary_df[
    configuration_summary_df["configuration_id"].isin(competitive_ids)
].copy()

selection_sort = [
    "method", "partial_mrr_at_10", "complete_recall_at_10",
    "mean_returned_tokens_at_5", "median_latency_at_5", "configuration_id"
]

competitive_configurations_df = competitive_configurations_df.sort_values(
    selection_sort,
    ascending=[True, False, False, True, True, True]
)

competitive_configurations_df["competitive_rank"] = (
    competitive_configurations_df.groupby("method").cumcount() + 1
)

selected_preview_df = (
    competitive_configurations_df[
        competitive_configurations_df["competitive_rank"] == 1
    ]
    .sort_values("method")
    .reset_index(drop=True)
)

assert len(selected_preview_df) == 5
assert selected_preview_df["method"].nunique() == 5

selection_columns = [
    "method", "configuration_id", "parameters_json",
    "complete_recall_at_5", "partial_mrr_at_10",
    "complete_recall_at_10", "mean_returned_tokens_at_5",
    "median_latency_at_5", "indexed_nodes"
]

display(selected_preview_df[selection_columns].style.format({
    "complete_recall_at_5": "{:.3f}",
    "partial_mrr_at_10": "{:.3f}",
    "complete_recall_at_10": "{:.3f}",
    "mean_returned_tokens_at_5": "{:,.1f}",
    "median_latency_at_5": "{:.3f}",
    "indexed_nodes": "{:,.0f}",
}))

,method,configuration_id,parameters_json,complete_recall_at_5,partial_mrr_at_10,complete_recall_at_10,mean_returned_tokens_at_5,median_latency_at_5,indexed_nodes
0,hierarchical,hierarchical_9c0a7874,"{""chunk_sizes"": [2048, 1024, 512]}",0.500,0.594,0.571,"2,426.4",0.392,"11,118"
1,semantic,semantic_b61156c9,"{""buffer_size"": 1, ""threshold"": 98}",0.643,0.572,0.714,"16,765.4",0.107,"2,557"
2,sentence,sentence_b9110a00,"{""chunk_overlap"": 50, ""chunk_size"": 512}",0.518,0.616,0.643,"2,538.9",0.508,"9,033"
3,sentence_window,sentence_window_ff7e26a2,"{""window_size"": 7}",0.643,0.721,0.661,"4,007.0",3.738,"106,509"
4,token,token_b9110a00,"{""chunk_overlap"": 50, ""chunk_size"": 512}",0.482,0.602,0.571,"2,840.1",0.297,"8,524"


In [110]:
# Clarify that this is the sum of separately stored node-token counts.
configuration_summary_df = configuration_summary_df.rename(columns={
    "mean_returned_tokens_at_5": "mean_summed_node_tokens_at_5"
})
competitive_configurations_df = competitive_configurations_df.rename(columns={
    "mean_returned_tokens_at_5": "mean_summed_node_tokens_at_5"
})
selected_preview_df = selected_preview_df.rename(columns={
    "mean_returned_tokens_at_5": "mean_summed_node_tokens_at_5"
})

In [111]:
# Preserve a transparent audit trail showing which configurations reached tie-breaks.
competitive_audit_columns = [
    "method", "configuration_id", "parameters_json",
    "complete_recall_at_5", "partial_mrr_at_10", "complete_recall_at_10",
    "mean_summed_node_tokens_at_5", "median_latency_at_5",
    "indexed_nodes", "competitive_rank"
]

display(
    competitive_configurations_df[competitive_audit_columns]
    .sort_values(["method", "competitive_rank"])
    .style.format({
        "complete_recall_at_5": "{:.3f}",
        "partial_mrr_at_10": "{:.3f}",
        "complete_recall_at_10": "{:.3f}",
        "mean_summed_node_tokens_at_5": "{:,.1f}",
        "median_latency_at_5": "{:.3f}",
        "indexed_nodes": "{:,.0f}",
    })
)

,method,configuration_id,parameters_json,complete_recall_at_5,partial_mrr_at_10,complete_recall_at_10,mean_summed_node_tokens_at_5,median_latency_at_5,indexed_nodes,competitive_rank
3,hierarchical,hierarchical_9c0a7874,"{""chunk_sizes"": [2048, 1024, 512]}",0.500,0.594,0.571,"2,426.4",0.392,"11,118",1
8,semantic,semantic_b61156c9,"{""buffer_size"": 1, ""threshold"": 98}",0.643,0.572,0.714,"16,765.4",0.107,"2,557",1
6,semantic,semantic_4a84176f,"{""buffer_size"": 3, ""threshold"": 95}",0.571,0.565,0.661,"7,571.7",0.211,"5,713",2
9,semantic,semantic_b70aa92a,"{""buffer_size"": 1, ""threshold"": 95}",0.554,0.554,0.696,"7,388.3",0.207,"5,713",3
14,sentence,sentence_b9110a00,"{""chunk_overlap"": 50, ""chunk_size"": 512}",0.518,0.616,0.643,"2,538.9",0.508,"9,033",1
12,sentence,sentence_4abeca5b,"{""chunk_overlap"": 100, ""chunk_size"": 1024}",0.554,0.615,0.679,"5,225.3",0.274,"4,462",2
10,sentence,sentence_2b17acfd,"{""chunk_overlap"": 200, ""chunk_size"": 2048}",0.607,0.501,0.714,"10,636.5",0.174,"2,269",3
19,sentence_window,sentence_window_ff7e26a2,"{""window_size"": 7}",0.643,0.721,0.661,"4,007.0",3.738,"106,509",1
24,token,token_b9110a00,"{""chunk_overlap"": 50, ""chunk_size"": 512}",0.482,0.602,0.571,"2,840.1",0.297,"8,524",1
22,token,token_4abeca5b,"{""chunk_overlap"": 100, ""chunk_size"": 1024}",0.607,0.566,0.661,"5,588.1",0.150,"4,314",2


In [112]:
# Freeze the rank-1 competitive configuration from each method.
frozen_configurations_df = (
    competitive_configurations_df[competitive_configurations_df["competitive_rank"].eq(1)]
    .sort_values("method").reset_index(drop=True).copy()
)

assert len(frozen_configurations_df) == 5
assert frozen_configurations_df["method"].nunique() == 5
assert frozen_configurations_df["configuration_id"].is_unique

display(frozen_configurations_df[competitive_audit_columns].style.format({
    "complete_recall_at_5": "{:.6f}", "partial_mrr_at_10": "{:.6f}",
    "complete_recall_at_10": "{:.6f}", "mean_summed_node_tokens_at_5": "{:,.1f}",
    "median_latency_at_5": "{:.3f}", "indexed_nodes": "{:,.0f}"
}))

,method,configuration_id,parameters_json,complete_recall_at_5,partial_mrr_at_10,complete_recall_at_10,mean_summed_node_tokens_at_5,median_latency_at_5,indexed_nodes,competitive_rank
0,hierarchical,hierarchical_9c0a7874,"{""chunk_sizes"": [2048, 1024, 512]}",0.500000,0.593622,0.571429,"2,426.4",0.392,"11,118",1
1,semantic,semantic_b61156c9,"{""buffer_size"": 1, ""threshold"": 98}",0.642857,0.571726,0.714286,"16,765.4",0.107,"2,557",1
2,sentence,sentence_b9110a00,"{""chunk_overlap"": 50, ""chunk_size"": 512}",0.517857,0.615795,0.642857,"2,538.9",0.508,"9,033",1
3,sentence_window,sentence_window_ff7e26a2,"{""window_size"": 7}",0.642857,0.721429,0.660714,"4,007.0",3.738,"106,509",1
4,token,token_b9110a00,"{""chunk_overlap"": 50, ""chunk_size"": 512}",0.482143,0.602154,0.571429,"2,840.1",0.297,"8,524",1


In [114]:
# Save the development-selection evidence and frozen manifest atomically.

SUMMARY_PATH = BENCHMARK_FOLDER / "development_configuration_summary_2.4.1.csv"
BOOTSTRAP_PATH = BENCHMARK_FOLDER / "development_paired_bootstrap_audit_2.4.1.csv"
FROZEN_PATH = BENCHMARK_FOLDER / "frozen_chunking_configurations_2.4.1.json"

bootstrap_audit_df = pd.DataFrame(bootstrap_rows)
assert len(configuration_summary_df) == 25
assert len(bootstrap_audit_df) == 25
assert len(frozen_configurations_df) == 5

def atomic_csv(dataframe, path):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    dataframe.to_csv(temporary_path, index=False)
    temporary_path.replace(path)

atomic_csv(configuration_summary_df, SUMMARY_PATH)
atomic_csv(bootstrap_audit_df, BOOTSTRAP_PATH)

# Convert parameters from JSON strings into reusable dictionaries.
frozen_export_df = frozen_configurations_df[[
    "method", "configuration_id", "parameters_json", "complete_recall_at_5",
    "partial_mrr_at_10", "complete_recall_at_10", "mean_summed_node_tokens_at_5",
    "median_latency_at_5", "indexed_nodes", "competitive_rank"
]].copy()
frozen_export_df["parameters"] = frozen_export_df.pop("parameters_json").apply(
    lambda value: json.loads(value) if isinstance(value, str) else value
)

source_hash = hashlib.sha256(CORRECTED_RESULTS_PATH.read_bytes()).hexdigest()
frozen_manifest = {
    "experiment_version": "2.4.1",
    "corpus_hash": "4cfb01ee22d2a6d039d64b1dc58be32815ce601aabbc01134cfcc130acaaeb59",
    "embedding_model": "BAAI/bge-m3",
    "development_questions": 56,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "source_results": CORRECTED_RESULTS_PATH.name,
    "source_results_sha256": source_hash,
    "selection_rule": [
        "highest complete Recall@5",
        "paired-bootstrap competitive set",
        "higher partial MRR@10",
        "higher complete Recall@10",
        "fewer summed node tokens at k=5",
        "lower median retrieval latency at k=5"
    ],
    "configurations": json.loads(frozen_export_df.to_json(orient="records"))
}

temporary_path = FROZEN_PATH.with_suffix(FROZEN_PATH.suffix + ".tmp")
temporary_path.write_text(json.dumps(frozen_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
temporary_path.replace(FROZEN_PATH)

print("Saved:", SUMMARY_PATH)
print("Saved:", BOOTSTRAP_PATH)
print("Saved:", FROZEN_PATH)

Saved: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_configuration_summary_2.4.1.csv
Saved: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_paired_bootstrap_audit_2.4.1.csv
Saved: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/frozen_chunking_configurations_2.4.1.json


In [115]:
# Reload saved outputs to ensure they are complete and readable.
saved_summary_df = pd.read_csv(SUMMARY_PATH)
saved_bootstrap_df = pd.read_csv(BOOTSTRAP_PATH)
saved_manifest = json.loads(FROZEN_PATH.read_text(encoding="utf-8"))

assert len(saved_summary_df) == 25
assert len(saved_bootstrap_df) == 25
assert len(saved_manifest["configurations"]) == 5
assert len({row["method"] for row in saved_manifest["configurations"]}) == 5
assert hashlib.sha256(CORRECTED_RESULTS_PATH.read_bytes()).hexdigest() == saved_manifest["source_results_sha256"]

print("Frozen development selection passed all checks.")
display(pd.DataFrame(saved_manifest["configurations"])[["method", "configuration_id", "parameters"]])

Frozen development selection passed all checks.


,method,configuration_id,parameters
0,hierarchical,hierarchical_9c0a7874,"{'chunk_sizes': [2048, 1024, 512]}"
1,semantic,semantic_b61156c9,"{'buffer_size': 1, 'threshold': 98}"
2,sentence,sentence_b9110a00,"{'chunk_overlap': 50, 'chunk_size': 512}"
3,sentence_window,sentence_window_ff7e26a2,{'window_size': 7}
4,token,token_b9110a00,"{'chunk_overlap': 50, 'chunk_size': 512}"
